# Settlement source: Kalshi daily highs versus NWS CLI

**Snapshot:** acquisition began 2026-11-12 23:50:13 UTC. Scope is all seven configured
stations, all API-served settled history, including legacy HIGH tickers returned through
canonical KX series filters. This notebook is research only and is never imported by `src/`.

**Question:** Do settled YES brackets disagree with IEM's stored NWS CLI maximum,
how do numeric `expiration_value` maxima differ, and when do individual market rules
first cite The Weather Company? Counts and source wording come from real responses.

The committed outputs are readable without the cache. Re-execution defaults to **offline**.
Cached response bodies are preserved byte-for-byte, base64-encoded inside gzipped
envelopes under `data/spike/settlement_source/`; this path is checked with `git check-ignore`
before each write. No collector `raw/` files or test fixtures are written.

Install optional dependencies with `uv sync --group notebooks`. Use the repo's Python 3.12
environment as the notebook kernel and run all cells. For an initial acquisition only,
start the kernel with `KWB_SPIKE_OFFLINE=0`. To obtain a different snapshot, move the
existing spike cache aside explicitly; ordinary reruns reuse its frozen responses.

Runtime dependencies are unchanged; pandas, nbformat, nbclient and ipykernel live in the
separate `notebooks` uv group. The HTTP client is Python's standard library.

## Findings for the frozen snapshot

**Yes: two bracket disagreements against IEM's stored CLI maximum.** Both are Miami:
`KXHIGHMIA-24NOV27-T83` (NWS period: CLI 81°F, settlement field 84°F, YES >83°F) and
`KXHIGHMIA-26AUG29-B90.5` (TWC period: CLI 85°F, settlement field 90°F, YES [90,91]°F).
The earlier case has a CLI revision **after** settlement. In the TWC case, the 85°F CLI
issuance preceded settlement; an earlier issuance reported 90°F. This establishes an
observed mismatch, not the cause of TWC's value.

All seven configured KX daily-high histories first cite TWC for climate day
**2026-08-14**; their last preceding NWS day is 2026-08-13. The change to a KX ticker in
2024 is therefore a different event from the source switch. Per-market primary rules,
including their verbatim wording, establish this result.

The dataset contains **47,895 unique markets and 8,525 events**. Live/historical endpoints
overlap on 252 tickers, with identical settlement-critical fields. All 8,525 events match
the independent `/events` inventory. Thirteen calendar gaps are listed explicitly below.

Bracket results: **KX 2/5,105 (0.0392%)**, legacy **HIGH 0/3,088** comparable events.
Within the TWC portion specifically, the rate is **1/630 (0.1587%)**. All other events
remain visible: 182 zero-YES and 138 multiple-YES events, one malformed early event date,
12 missing IEM maxima, and two fractional market payouts. Most nonunique-YES cases are
early legacy threshold events, so a single-winner bracket interpretation does not apply.

Numeric `expiration_value` is populated on **24,602/30,696 KX** and **3,718/17,199 HIGH**
market records. After joining available CLI data and excluding the date conflict, the
**event-weighted** exact differences (settlement field minus IEM high, °F) are:

| Source from each market's primary rules | −2 | −1 | 0 | +1 | +2 | +3 | +5 |
|---|---:|---:|---:|---:|---:|---:|---:|
| KX citing NWS CLI | 0 | 4 | 4,007 | 3 | 0 | 1 | 0 |
| KX citing TWC | 0 | 0 | 628 | 0 | 0 | 0 | 1 |
| Legacy HIGH | 1 | 4 | 3,400 | 1 | 1 | 0 | 0 |

| Cohort | Events | Bracket agreement | Exact agreement |
|---|---:|---:|---:|
| KX citing NWS CLI | 4,486 | 4,474/4,475 | 4,007/4,015 |
| KX citing TWC | **630** | **629/630** | **628/629** |
| Legacy HIGH | 3,409 | 3,088/3,088 | 3,400/3,407 |

The TWC row is the direct label-source comparison. Every nonzero TWC difference is
listed below: only `KXHIGHMIA-26AUG29`, with expiration 90°F versus IEM 85°F (+5°F).
The remaining TWC event, `KXHIGHLAX-26OCT22`, has the textual expiration “70° to 71°”
and therefore has no numeric exact comparison. Legacy HIGH remains a family control;
475 older events do not identify a source in `rules_primary`, which is kept explicit.

These are differences against **stored IEM values**, not a claim that each IEM value is
a completed final daily report. Original-text checks found eight latest linked products
that are preliminary same-day reports; bounded searches found completed numeric products
for four of them, and none for the other four. Two KX +1°F differences involve those
unresolved preliminary labels. One IEM row's stored maximum (Philadelphia 2026-08-18,
88°F) differs from its linked preliminary text (87°F); a completed archived product
corroborates 88°F. These cases stay explicit instead of being silently repaired.

One legacy numeric expiration is itself inconsistent: `HIGHCHI-21OCT01-T84` reports 84°F
while settling YES to >84°F; CLI says 86°F. All eleven nonnumeric expiration values are
listed. Exact fields are valuable evidence, but this anomaly prevents treating every
populated field as reliable exact settlement truth.

**Recorded labels policy:** Kalshi's settled result wherever a market exists, IEM's stored CLI high otherwise. IEM's latest issuance can differ from the issuance available at settlement; the source-specific evidence and exceptions remain explicit.

**Decision appended to `docs/decisions.md`:**

> 2026-11-12: Refines the 2026-11-07 settlement-source decision: all seven configured KX daily-high series cited NWS CLI through climate day 2026-08-13 and TWC from 2026-08-14; across 630 TWC events, TWC versus IEM CLI agrees on 629/630 YES brackets and 628/629 numeric exact maxima (one +5°F difference, KXHIGHMIA-26AUG29; one nonnumeric expiration excluded from exact comparison); legacy HIGH control agrees on 3088/3088 comparable brackets. Labels policy: Kalshi's settled result wherever a market exists, IEM's stored CLI high otherwise. Known limitation: IEM stores the latest issuance, which can differ from what was published at settlement (KXHIGHMIA-24NOV27 settled at 84°F before the CLI revised to 81°F). Evidence: notebooks/01_settlement_source.ipynb.

**Design choices:** Event weighting avoids counting the same weather outcome six times.
Both endpoints are read because observed archival overlap makes exclusive cutoff routing
unnecessary and brittle. Rule text supplies old missing strikes and authoritative climate
dates, while conflicting metadata remains an anomaly. Annual IEM responses remain the
baseline so that archive-quality findings cannot silently change the measured label source. Accounting distinguishes actual settled events from missing calendar slots and assigns overlapping exclusions only once.

**Out of scope, left unchanged:** `plan.md` still calls spike item 1 “Bracket structure”;
the request and first open decision define this settlement-source investigation. Existing
unrelated working-tree changes were left untouched. No source modules, collector raw data,
test fixtures were changed. The approved decision was appended; nothing was committed or pushed.


## Provider documentation verified before client code

- [Kalshi historical-data guide](https://docs.kalshi.com/getting_started/historical_data)
  and [cutoff endpoint](https://docs.kalshi.com/api-reference/historical/get-historical-cutoff-timestamps.md):
  `market_settled_ts` partitions by settlement time, not climate day.
- [Historical markets](https://docs.kalshi.com/api-reference/historical/get-historical-markets):
  public GET, cursor pagination up to 1,000 records; select one of `series_ticker`,
  `event_ticker`, `tickers`. No historical timestamp filter is assumed.
- [Live settled markets](https://docs.kalshi.com/api-reference/market/get-markets.md):
  `status=settled`, `series_ticker`, `limit`, `cursor`; `result`, `expiration_value`,
  `settlement_ts`, `strike_type`, `floor_strike`, `cap_strike`, `rules_primary`.
  Response lifecycle statuses include `finalized` even though the query filter is `settled`.
  `settlement_bounds_type` describes dollar payout bounds, not temperatures.
- [Events](https://docs.kalshi.com/api-reference/events/get-events.md): independent,
  paginated event inventory includes archived events. Its date-based `strike_date`
  need not have the climate day's UTC date. Join dates come from each market's own
  primary rules, cross-checked against event text and ticker. Nested markets omit archives.
- [Series discovery](https://docs.kalshi.com/api-reference/market/get-series-list.md):
  query unfiltered; city names select candidates only. Station identities require
  rule/source evidence matched to `config/nws_stations.csv`.
- [Kalshi limits](https://docs.kalshi.com/getting_started/rate_limits): documented limits
  are token budgets for authenticated requests. This public client follows the stricter
  recorded spacing: `/markets` starts at least 0.25 s apart; other Kalshi requests 0.1 s.
- [IEM CLI help](https://mesonet.agron.iastate.edu/json/cli.py?help) and
  [endpoint implementation](https://github.com/akrherz/iem/blob/main/pylib/iemweb/json/cli.py):
  query station/year; `high` is integer Fahrenheit or `M`; `valid` is the climate-summary
  date; `product` and `link` identify the referenced text.
- [IEM CLI parser](https://github.com/akrherz/pyIEM/blob/main/src/pyiem/nws/products/cli.py):
  the headline supplies the climate date; newer products replace a station/day row.
  Therefore a row may be preliminary or revised, rather than an immutable final label.
- [CLI audit](https://mesonet.agron.iastate.edu/json/cli_audit.py?help) and
  [original-text search](https://mesonet.agron.iastate.edu/json/nwstext_search.py?help)
  support issuance investigations. Text-search windows use UTC only to find products;
  the report headline still determines the climate day.
- [IEM API overview](https://mesonet.agron.iastate.edu/api/) and
  [usage policy](https://mesonet.agron.iastate.edu/disclaimer.php): no numeric CLI request
  limit was found. Our conservative etiquette is serial requests at least one second
  apart, annual bulk queries, exact-body caching, and bounded backoff.

Bound rules: strict tails remain strict; `greater_or_equal` / `less_or_equal` retain
equality. Finite ranges use the documented minimum/maximum YES values and the market's
explicit range wording. Old records missing structured strikes use their explicit
primary-rule interval, with the evidence path counted. No ticker strike is used as a bound.

## Setup and exact-response cache

In [1]:
import base64
import csv
import gzip
import hashlib
import json
import os
import re
import subprocess
import time
from collections import Counter, defaultdict
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal, InvalidOperation
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlsplit
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config/nws_stations.csv").is_file(), "Run from repo root or notebooks/"
CACHE = ROOT / "data/spike/settlement_source"
OFFLINE = os.environ.get("KWB_SPIKE_OFFLINE", "1") == "1"
BASE = "https://external-api.kalshi.com/trade-api/v2"


def ignored_write(path, body):
    relative = path.relative_to(ROOT)
    check = subprocess.run(
        ["git", "check-ignore", str(relative)], cwd=ROOT, capture_output=True, text=True
    )
    assert check.returncode == 0, f"Cache path is not ignored: {relative}"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(body)


META_PATH = CACHE / "snapshot.json"
if META_PATH.exists():
    SNAPSHOT = json.loads(META_PATH.read_text())
else:
    assert not OFFLINE, "No cache. Set KWB_SPIKE_OFFLINE=0 for the initial acquisition."
    SNAPSHOT = {"started_at": datetime.now(UTC).isoformat(), "schema": 1}
    ignored_write(META_PATH, json.dumps(SNAPSHOT, indent=2).encode())
AS_OF = datetime.fromisoformat(SNAPSHOT["started_at"])
CACHE_DAY = AS_OF.date().isoformat()
with (ROOT / "config/nws_stations.csv").open(newline="") as stream:
    STATIONS = list(csv.DictReader(stream))
assert len(STATIONS) == 7
CONFIG_BY_CLI = {row["cli_code"]: row for row in STATIONS}
CONFIG_BY_STATION = {row["station_id"]: row for row in STATIONS}
assert len(CONFIG_BY_CLI) == len(CONFIG_BY_STATION) == 7
REQUESTS = []
NEXT_REQUEST = defaultdict(float)


def cached_response(url, params=None, *, tag=None, allow_status=()):
    """Archive exact bytes before JSON/text decoding; no src/ collector imports."""
    params = {} if params is None else params
    full_url = url + ("?" + urlencode(sorted(params.items())) if params else "")
    source = "kalshi" if url.startswith(BASE + "/") else "iem"
    key = hashlib.sha256((full_url + "|" + (tag or "")).encode()).hexdigest()
    filename = "cutoff.json.gz" if tag == "cutoff_start" else key + ".json.gz"
    path = CACHE / source / CACHE_DAY / filename
    if path.exists():
        envelope = json.loads(gzip.decompress(path.read_bytes()))
        assert envelope["request_url"] == full_url
        body = base64.b64decode(envelope["body_base64"], validate=True)
        assert hashlib.sha256(body).hexdigest() == envelope["body_sha256"]
        status = envelope["http_status"]
        REQUESTS.append({"url": full_url, "cache": True, "path": str(path.relative_to(ROOT))})
    else:
        assert not OFFLINE, f"Missing cached response: {full_url} ({tag=})"
        is_markets = urlsplit(url).path == "/trade-api/v2/markets"
        interval = (0.25 if is_markets else 0.1) if source == "kalshi" else 1.0
        for attempt in range(5):
            ready = max(NEXT_REQUEST[source], NEXT_REQUEST["markets"] if is_markets else 0)
            time.sleep(max(0, ready - time.monotonic()))
            requested_at = datetime.now(UTC)
            NEXT_REQUEST[source] = time.monotonic() + (0.1 if source == "kalshi" else 1.0)
            if is_markets:
                NEXT_REQUEST["markets"] = time.monotonic() + interval
            request = Request(
                full_url,
                headers={
                    "User-Agent": "Kalshi-Weather-Bot/0.1 (cached settlement-source research)",
                    "Accept": "application/json, text/plain;q=0.9, */*;q=0.1",
                },
            )
            try:
                try:
                    response = urlopen(request, timeout=90)
                except HTTPError as error:
                    response = error
                with response:
                    body = response.read()
                    status = response.status
                    headers = dict(response.headers.items())
                received_at = datetime.now(UTC)
            except (OSError, URLError):
                if attempt == 4:
                    raise
                time.sleep(2 ** (attempt + 1))
                continue
            envelope = {
                "request_url": full_url,
                "request_params": params,
                "requested_at": requested_at.isoformat(),
                "received_at": received_at.isoformat(),
                "http_status": status,
                "response_headers": headers,
                "body_base64": base64.b64encode(body).decode("ascii"),
                "body_sha256": hashlib.sha256(body).hexdigest(),
            }
            archived_path = (
                path
                if 200 <= status < 300 or status in allow_status
                else (path.with_name(key + f".attempt-{attempt}.json.gz"))
            )
            ignored_write(archived_path, gzip.compress(json.dumps(envelope).encode()))
            REQUESTS.append(
                {"url": full_url, "cache": False, "path": str(archived_path.relative_to(ROOT))}
            )
            if 200 <= status < 300 or status in allow_status:
                break
            if status not in (408, 429) and status < 500:
                raise RuntimeError(f"HTTP {status}: {full_url}: {body[:300]!r}")
            retry_after = headers.get("Retry-After", "")
            time.sleep(
                max(float(retry_after) if retry_after.isdecimal() else 0, 2 ** (attempt + 1))
            )
        else:
            raise RuntimeError(f"Retries exhausted: {full_url}")
    assert 200 <= status < 300 or status in allow_status, f"HTTP {status}: {full_url}"
    return body, status, str(path.relative_to(ROOT))


def get_json(path, params=None, *, tag=None):
    url = path if path.startswith("https://") else BASE + path
    body, status, provenance = cached_response(url, params, tag=tag)
    assert status == 200
    value = json.loads(body)
    assert isinstance(value, dict), (url, type(value))
    return value, provenance


PAGES = []


def paginate(path, params, field):
    cursor = ""
    seen = set()
    records = []
    while True:
        query = {**params, **({"cursor": cursor} if cursor else {})}
        body, provenance = get_json(path, query)
        page_records = body[field]
        assert isinstance(page_records, list)
        next_cursor = body.get("cursor", "")
        assert isinstance(next_cursor, str)
        PAGES.append(
            {
                "path": path,
                "series": params.get("series_ticker"),
                "cursor_in": cursor,
                "cursor_out": next_cursor,
                "count": len(page_records),
                "cache_path": provenance,
                "tickers": [row.get("ticker", row.get("event_ticker")) for row in page_records],
            }
        )
        records.extend(
            {**row, "_endpoint": path, "_cache_path": provenance} for row in page_records
        )
        if not next_cursor:
            break
        assert next_cursor not in seen, f"Pagination cursor loop: {path} {params}"
        seen.add(next_cursor)
        cursor = next_cursor
    return records


def show_table(frame):
    """Render complete tables, including exceptions; never silently truncate rows."""
    if not isinstance(frame, pd.DataFrame):
        frame = pd.DataFrame(frame)
    if frame.empty:
        print("(none)")
    else:
        display(HTML(frame.to_html(index=False, max_rows=None, max_cols=None, render_links=True)))


print(f"Snapshot starts {AS_OF.isoformat()}; offline={OFFLINE}; cache={CACHE.relative_to(ROOT)}")

Snapshot starts 2026-11-12T23:50:13.180836+00:00; offline=True; cache=data/spike/settlement_source


## Full acquisition across the cutoff

Live pages are read before historical pages, then the cutoff is checked again. Every
pagination chain must terminate. Overlaps are kept in raw provenance and deduplicated
by ticker only after settlement-critical fields agree. Differences in non-settlement
fields are recorded. `/events` is the independent completeness check; there is no second
market census. Counts cover API-served history, not records the provider no longer serves.

In [2]:
CUTOFF_START, _ = get_json("/historical/cutoff", tag="cutoff_start")
series_payload, _ = get_json("/series")
ALL_SERIES = {row["ticker"]: row for row in series_payload["series"]}

# City names select candidates only. A market's CLI identifier must independently
# match the checked-in mapping before it can contribute to a station comparison.
CITY_NAMES = {
    "KAUS": ["austin"],
    "KMDW": ["chicago"],
    "KDEN": ["denver"],
    "KLAX": ["los angeles"],
    "KMIA": ["miami"],
    "KNYC": ["nyc", "new york city"],
    "KPHL": ["philadelphia"],
}
LEGACY_CANDIDATES = [
    "HIGHAUS",
    "HIGHCHI",
    "HIGHDEN",
    "HIGHLAX",
    "HIGHMIA",
    "HIGHNY",
    "HIGHNY0",
    "HIGHPHIL",
]


def candidate_station(series):
    title = series["title"].casefold()
    words = set(re.findall(r"[a-z]+", title))
    if series["frequency"].casefold() != "daily":
        return None
    if not words & {"temperature", "temperatures", "temp"}:
        return None
    if words & {"low", "lowest", "min", "minimum", "difference", "change", "anomaly"}:
        return None
    if not words & {"high", "highest", "maximum", "max"}:
        return None
    matches = [
        station
        for station, names in CITY_NAMES.items()
        if any(re.search(r"\b" + re.escape(name) + r"\b", title) for name in names)
    ]
    assert len(matches) <= 1, series
    return matches[0] if matches else None


CANDIDATES = {
    ticker: candidate_station(series)
    for ticker, series in ALL_SERIES.items()
    if ticker.startswith(("KX", "HIGH")) and candidate_station(series)
}
for row in STATIONS:
    CANDIDATES[row["series_ticker"]] = row["station_id"]
# Probe known legacy names even if discovery no longer lists them. These are not
# station mappings: matching uses a configured CLI code in each market's rules.
for ticker in LEGACY_CANDIDATES:
    CANDIDATES.setdefault(ticker, None)

LIVE = {}
HISTORICAL = {}
EVENT_INVENTORY = {}
for ticker in sorted(CANDIDATES):
    LIVE[ticker] = paginate(
        "/markets",
        {
            "series_ticker": ticker,
            "status": "settled",
            "limit": 1000,
        },
        "markets",
    )
    HISTORICAL[ticker] = paginate(
        "/historical/markets",
        {
            "series_ticker": ticker,
            "limit": 1000,
        },
        "markets",
    )
    EVENT_INVENTORY[ticker] = paginate(
        "/events",
        {
            "series_ticker": ticker,
            "status": "settled",
            "limit": 200,
        },
        "events",
    )
    print(
        ticker,
        "live",
        len(LIVE[ticker]),
        "historical",
        len(HISTORICAL[ticker]),
        "events",
        len(EVENT_INVENTORY[ticker]),
        flush=True,
    )
CUTOFF_END, _ = get_json("/historical/cutoff", tag="cutoff_end")
assert CUTOFF_START["market_settled_ts"] == CUTOFF_END["market_settled_ts"], (
    "The cutoff moved during acquisition. Reconcile affected events before claiming completeness.",
    CUTOFF_START,
    CUTOFF_END,
)
print("Stable market cutoff:", CUTOFF_START["market_settled_ts"])

ALL_FETCHED = [row for ticker in sorted(CANDIDATES) for row in LIVE[ticker] + HISTORICAL[ticker]]
BY_TICKER = defaultdict(list)
for row in ALL_FETCHED:
    BY_TICKER[row["ticker"]].append(row)
MARKETS = []
OVERLAPS = []
AFTER_SNAPSHOT = []
for ticker, copies in BY_TICKER.items():
    if len(copies) > 1:
        clean = [
            {key: value for key, value in row.items() if not key.startswith("_")} for row in copies
        ]
        critical = [
            "ticker",
            "event_ticker",
            "market_type",
            "result",
            "settlement_ts",
            "expiration_value",
            "strike_type",
            "floor_strike",
            "cap_strike",
            "rules_primary",
            "rules_secondary",
            "title",
            "yes_sub_title",
        ]
        assert all(all(row.get(key) == clean[0].get(key) for key in critical) for row in clean), (
            f"Conflicting settlement copies: {ticker}"
        )
        differences = {
            key: [row.get(key) for row in clean]
            for key in set().union(*(row.keys() for row in clean))
            if any(row.get(key) != clean[0].get(key) for row in clean)
        }
        OVERLAPS.append(
            {
                "ticker": ticker,
                "endpoints": [row["_endpoint"] for row in copies],
                "non_settlement_differences": differences,
            }
        )
    row = copies[0]
    settled_at = row.get("settlement_ts")
    if settled_at and datetime.fromisoformat(settled_at.replace("Z", "+00:00")) > AS_OF:
        AFTER_SNAPSHOT.append(ticker)
    else:
        MARKETS.append(row)

print(
    "Unique markets:",
    len(MARKETS),
    "overlap:",
    len(OVERLAPS),
    "settled after snapshot:",
    len(AFTER_SNAPSHOT),
)
print("Status values:", Counter(row["status"] for row in MARKETS))
print("Result values:", Counter(row["result"] for row in MARKETS))
print("Strike types:", Counter(row.get("strike_type") for row in MARKETS))

HIGHAUS live 0 historical 0 events 0


HIGHCHI live 0 historical 0 events 0


HIGHDEN live 0 historical 0 events 0


HIGHLAX live 0 historical 0 events 0


HIGHMIA live 0 historical 0 events 0


HIGHNY live 0 historical 0 events 0


... (9 outputs left out)

KXHIGHNY0 live 0 historical 21 events 20


KXHIGHPHIL live 402 historical 3966 events 722


Stable market cutoff: 2026-09-13T00:00:00Z
Unique markets: 47895 overlap: 252 settled after snapshot: 0
Status values: Counter({'finalized': 47895})


Result values: Counter({'no': 39412, 'yes': 8481, 'scalar': 2})
Strike types: Counter({'between': 29942, 'less': 7511, 'greater': 7471, None: 2971})


## Identity, climate date, source and interval parsing

The checked-in station/CLI mapping is authoritative. Historical event source URLs and
explicit station names corroborate it; no station is inferred from ticker letters.
Source classification uses **only that market's verbatim `rules_primary`**. A source
omitted there is recorded as “Not stated,” even when secondary rules identify NWS.

Known data inconsistencies are retained as anomalies. A date conflict prevents a reliable
bracket comparison. Textual expiration values remain textual; scalar settlements remain
visible. None is silently converted to a temperature or dropped from the population.

In [3]:
from urllib.parse import parse_qs

MONTH_NUMBER = {
    name.casefold(): i
    for i, names in enumerate(
        [
            [],
            ["Jan", "January"],
            ["Feb", "February"],
            ["Mar", "March"],
            ["Apr", "April"],
            ["May"],
            ["Jun", "June"],
            ["Jul", "July"],
            ["Aug", "August"],
            ["Sep", "Sept", "September"],
            ["Oct", "October"],
            ["Nov", "November"],
            ["Dec", "December"],
        ]
    )
    for name in names
}
DATE_RE = re.compile(
    r"\b(" + "|".join(MONTH_NUMBER) + r")\.?\s+(\d{1,2})(?:st|nd|rd|th)?,?\s+(20\d{2})\b",
    re.IGNORECASE,
)


def text_dates(text):
    return {
        date(int(year), MONTH_NUMBER[month.casefold()], int(day))
        for month, day, year in DATE_RE.findall(text)
    }


def ticker_date(event_ticker):
    suffix = event_ticker.rsplit("-", 1)[1]
    match = re.fullmatch(r"(\d{2})([A-Z]{3})(\d{2})", suffix)
    assert match, f"Unrecognized event date: {event_ticker}"
    year, month, day = match.groups()
    return date(2000 + int(year), MONTH_NUMBER[month.casefold()], int(day))


def climate_date(market, event):
    """The primary rule is authoritative; ticker is an independent check only."""
    dates = text_dates(market["rules_primary"])
    assert len(dates) == 1, f"Expected one rule climate date: {market['ticker']}: {dates}"
    result = next(iter(dates))
    for field in ["title", "yes_sub_title"]:
        other = text_dates(market.get(field, ""))
        assert not other or other == {result}, (market["ticker"], field, other, result)
    for field in ["title", "sub_title"]:
        other = text_dates(event.get(field, ""))
        assert not other or other == {result}, (market["ticker"], field, other, result)
    assert ticker_date(market["event_ticker"]) == result, market["ticker"]
    return result


def classify_source(rules_primary):
    lower = rules_primary.casefold()
    if "the weather company" in lower:
        return "The Weather Company"
    if "national weather service" in lower or re.search(r"\bnws\b", lower):
        return "NWS CLI"
    return "Not stated in rules_primary"


# Exact location clauses seen in old rules. They corroborate the per-event CLI
# source URL; city ticker letters are never consulted for station identity.
STATION_ALIASES = {
    "KAUS": ["austin bergstrom"],
    "KMDW": ["chicago midway", "midway international airport"],
    "KDEN": ["denver, co", "denver international"],
    "KLAX": ["los angeles airport", "los angeles international"],
    "KMIA": ["miami international airport", "miami, fl"],
    "KNYC": ["central park"],
    "KPHL": ["philadelphia international airport", "philadelphia, pa"],
}


def station_from_evidence(market, event):
    primary = market["rules_primary"]
    primary_codes = set(re.findall(r"\bCLI[A-Z0-9]{3}\b", primary))
    assert primary_codes <= CONFIG_BY_CLI.keys(), (market["ticker"], primary_codes)
    primary_matches = {CONFIG_BY_CLI[code]["station_id"] for code in primary_codes}
    secondary_codes = set(re.findall(r"\bCLI[A-Z0-9]{3}\b", market["rules_secondary"]))
    assert secondary_codes <= CONFIG_BY_CLI.keys(), (market["ticker"], secondary_codes)
    secondary_matches = {CONFIG_BY_CLI[code]["station_id"] for code in secondary_codes}
    url_matches = set()
    for source in event.get("settlement_sources", []):
        query = parse_qs(urlsplit(source["url"]).query)
        for issuedby in query.get("issuedby", []):
            code = "CLI" + issuedby.upper()
            if code in CONFIG_BY_CLI:
                url_matches.add(CONFIG_BY_CLI[code]["station_id"])
    clause = primary.split(" for ")[0].casefold()
    name_matches = {
        station
        for station, names in STATION_ALIASES.items()
        if any(name in clause for name in names)
    }
    evidence = primary_matches | secondary_matches | url_matches | name_matches
    assert len(evidence) == 1, (
        market["ticker"],
        evidence,
        primary,
        event.get("settlement_sources"),
    )
    station = next(iter(evidence))
    expected = CANDIDATES.get(event["series_ticker"])
    assert expected is None or station == expected, (market["ticker"], station, expected)
    return station, {
        "primary_cli": sorted(primary_codes),
        "secondary_cli": sorted(secondary_codes),
        "event_cli_url": sorted(url_matches),
        "rule_location_name": sorted(name_matches),
    }


def decimal_value(value, *, missing_allowed=False):
    if missing_allowed and value in ("", None):
        return None
    assert not isinstance(value, bool)
    try:
        result = Decimal(str(value))
    except InvalidOperation as error:
        raise AssertionError(f"Not a numeric value: {value!r}") from error
    assert result.is_finite(), value
    return result


NUMBER = r"(-?\d+(?:\.\d+)?)"


def interval_from_rules(primary):
    lower = primary.casefold()
    between = re.search(r"is between\s+" + NUMBER + r"\s*(?:[-–]|and)\s*" + NUMBER, lower)
    if between:
        return Decimal(between[1]), Decimal(between[2]), True, True
    for phrase, closed, side in [
        ("greater than or equal to", True, "lower"),
        ("less than or equal to", True, "upper"),
        ("strictly greater than", False, "lower"),
        ("strictly less than", False, "upper"),
        ("greater than", False, "lower"),
        ("less than", False, "upper"),
        ("above", False, "lower"),
        ("below", False, "upper"),
        ("at least", True, "lower"),
        ("at most", True, "upper"),
    ]:
        match = re.search(r"is\s+" + phrase + r"\s+" + NUMBER, lower)
        if match:
            value = Decimal(match[1])
            return (value, None, closed, False) if side == "lower" else (None, value, False, closed)
    raise AssertionError(f"Unrecognized interval wording: {primary}")


def interval_from_market(market):
    kind = market.get("strike_type")
    lower = decimal_value(market.get("floor_strike"), missing_allowed=True)
    upper = decimal_value(market.get("cap_strike"), missing_allowed=True)
    if kind is None:
        # Older served records omit structured strikes. Their verbatim rule
        # still specifies the interval; report this evidence path explicitly.
        interval = interval_from_rules(market["rules_primary"])
        assert lower is None or lower == interval[0]
        assert upper is None or upper == interval[1]
        return interval
    assert kind in {"greater", "greater_or_equal", "less", "less_or_equal", "between"}, (
        market["ticker"],
        kind,
    )
    if kind.startswith("greater"):
        assert lower is not None and upper is None
        interval = (lower, None, kind == "greater_or_equal", False)
    elif kind.startswith("less"):
        assert lower is None and upper is not None
        interval = (None, upper, False, kind == "less_or_equal")
    else:
        assert lower is not None and upper is not None and lower <= upper
        interval = (lower, upper, True, True)
    assert interval == interval_from_rules(market["rules_primary"]), (
        market["ticker"],
        interval,
        market["rules_primary"],
    )
    return interval


def contains(value, interval):
    lower, upper, lower_closed, upper_closed = interval
    return (lower is None or value > lower or (lower_closed and value == lower)) and (
        upper is None or value < upper or (upper_closed and value == upper)
    )


def bracket_label(interval):
    lower, upper, lower_closed, upper_closed = interval
    if lower is None:
        return f"x {'≤' if upper_closed else '<'} {upper}°F"
    if upper is None:
        return f"x {'≥' if lower_closed else '>'} {lower}°F"
    return f"{'[' if lower_closed else '('}{lower}, {upper}{']' if upper_closed else ')'}°F"


EVENTS = {}
for queried_series, records in EVENT_INVENTORY.items():
    for event in records:
        assert event["series_ticker"] == queried_series, (queried_series, event)
        assert event["event_ticker"] not in EVENTS, event["event_ticker"]
        EVENTS[event["event_ticker"]] = event

NORMALIZED = []
PARSE_ANOMALIES = []
for market in MARKETS:
    try:
        event = EVENTS[market["event_ticker"]]
        station, station_evidence = station_from_evidence(market, event)
        date_error = None
        try:
            day = climate_date(market, event)
        except AssertionError as error:
            dates = text_dates(market["rules_primary"])
            assert len(dates) == 1, market["ticker"]
            day = next(iter(dates))
            date_error = str(error)
        interval_error = None
        try:
            interval = interval_from_market(market)
        except AssertionError as error:
            interval = interval_from_rules(market["rules_primary"])
            interval_error = str(error)
        assert market["market_type"] == "binary"
        assert market["result"] in {"yes", "no", "scalar"}, market["result"]
        expiration_raw = market.get("expiration_value")
        expiration_error = None
        try:
            expiration = decimal_value(expiration_raw, missing_allowed=True)
        except AssertionError as error:
            expiration = None
            expiration_error = str(error)
        NORMALIZED.append(
            {
                "ticker": market["ticker"],
                "event_ticker": market["event_ticker"],
                "series": market["event_ticker"].rsplit("-", 1)[0],
                "retrieval_series": event["series_ticker"],
                "family": "KX" if market["ticker"].startswith("KX") else "HIGH",
                "station": station,
                "cli_code": CONFIG_BY_STATION[station]["cli_code"],
                "climate_date": day,
                "result": market["result"],
                "interval": interval,
                "date_error": date_error,
                "interval_error": interval_error,
                "bracket": bracket_label(interval),
                "expiration_value": expiration,
                "expiration_raw": expiration_raw,
                "expiration_error": expiration_error,
                "bounds_evidence": (
                    "rules_primary"
                    if market.get("strike_type") is None
                    else "structured fields + rules_primary"
                ),
                "source": classify_source(market["rules_primary"]),
                "rules_primary": market["rules_primary"],
                "rules_secondary": market["rules_secondary"],
                "station_evidence": station_evidence,
                "cache_path": market["_cache_path"],
                "endpoint": market["_endpoint"],
            }
        )
    except (AssertionError, KeyError, ValueError) as error:
        PARSE_ANOMALIES.append(
            {
                "ticker": market["ticker"],
                "error": str(error),
                "rules_primary": market["rules_primary"],
                "cache_path": market["_cache_path"],
            }
        )

print("Boundary/identity parse anomalies:", len(PARSE_ANOMALIES))
show_table(PARSE_ANOMALIES)
print("Parsed rows:", len(NORMALIZED))
print("Sources:", Counter(row["source"] for row in NORMALIZED))
print("Market prefixes:", Counter(row["series"] for row in NORMALIZED))
# Do not drop unresolved boundary records to obtain a clean comparison.
assert not PARSE_ANOMALIES, "Investigate every parse anomaly before comparisons."

Boundary/identity parse anomalies: 0
(none)
Parsed rows: 47895
Sources: Counter({'NWS CLI': 43385, 'The Weather Company': 3780, 'Not stated in rules_primary': 730})
Market prefixes: Counter({'HIGHNY': 5416, 'HIGHCHI': 5407, 'KXHIGHAUS': 4494, 'KXHIGHCHI': 4494, 'KXHIGHMIA': 4494, 'KXHIGHNY': 4494, 'KXHIGHDEN': 4332, 'KXHIGHPHIL': 4332, 'KXHIGHLAX': 4056, 'HIGHMIA': 3184, 'HIGHAUS': 3165, 'HIGHNY0': 21, 'HIGHCHI-2': 6})


## Comparison and complete exception tables

Rates use comparable **events**, not six repeated bracket records as six independent
weather observations. The denominator requires one YES bracket, a numeric CLI maximum,
and a consistent climate date. All other events remain in coverage/anomaly tables.
Early legacy events can be independent thresholds rather than exhaustive partitions.

Exact differences are `expiration_value − IEM high` in °F, computed with `Decimal`.
Both event counts and market-record counts are shown. Every numeric market value is
checked against the event's winning YES bracket, including values attached to NO markets.

In [4]:
GROUPED_MARKETS = defaultdict(list)
for row in NORMALIZED:
    GROUPED_MARKETS[row["event_ticker"]].append(row)

EVENT_ROWS = []
EVENT_ANOMALIES = []
MARKET_ANOMALIES = []
for event_ticker, rows in sorted(GROUPED_MARKETS.items()):
    yes_rows = [row for row in rows if row["result"] == "yes"]
    first = rows[0]
    identities = {(row["station"], row["climate_date"]) for row in rows}
    assert len(identities) == 1, (event_ticker, identities)
    sources = {row["source"] for row in rows}
    assert len(sources) == 1, (event_ticker, sources)
    yes = yes_rows[0] if len(yes_rows) == 1 else None
    if len(yes_rows) != 1:
        EVENT_ANOMALIES.append(
            {
                "event_ticker": event_ticker,
                "climate_date": first["climate_date"],
                "yes_count": len(yes_rows),
                "market_count": len(rows),
                "reason": "Event does not have exactly one YES bracket",
            }
        )
    numeric_values = {
        row["expiration_value"] for row in rows if row["expiration_value"] is not None
    }
    if len(numeric_values) > 1:
        EVENT_ANOMALIES.append(
            {
                "event_ticker": event_ticker,
                "climate_date": first["climate_date"],
                "reason": "Numeric expiration values differ within the event",
                "values": sorted(map(str, numeric_values)),
            }
        )
    for row in rows:
        for field in ["date_error", "interval_error", "expiration_error"]:
            if row[field]:
                MARKET_ANOMALIES.append(
                    {
                        "ticker": row["ticker"],
                        "climate_date": row["climate_date"],
                        "category": field,
                        "detail": row[field],
                        "expiration_raw": row["expiration_raw"],
                    }
                )
        if row["result"] == "scalar":
            MARKET_ANOMALIES.append(
                {
                    "ticker": row["ticker"],
                    "climate_date": row["climate_date"],
                    "category": "scalar result",
                    "detail": "Settled at a fractional payout, not binary YES/NO",
                }
            )
        if (
            yes
            and row["expiration_value"] is not None
            and not contains(row["expiration_value"], yes["interval"])
        ):
            MARKET_ANOMALIES.append(
                {
                    "ticker": row["ticker"],
                    "climate_date": row["climate_date"],
                    "category": "expiration outside YES bracket",
                    "detail": f"{row['expiration_value']} outside {yes['bracket']}",
                }
            )
    EVENT_ROWS.append(
        {
            "event_ticker": event_ticker,
            "series": first["series"],
            "retrieval_series": first["retrieval_series"],
            "family": first["family"],
            "station": first["station"],
            "cli_code": first["cli_code"],
            "climate_date": first["climate_date"],
            "source": first["source"],
            "market_count": len(rows),
            "yes_count": len(yes_rows),
            "yes_ticker": yes["ticker"] if yes else None,
            "yes_bracket": yes["bracket"] if yes else None,
            "yes_interval": yes["interval"] if yes else None,
            "expiration_value": next(iter(numeric_values)) if len(numeric_values) == 1 else None,
            "date_anomaly": any(row["date_error"] is not None for row in rows),
            "bounds_anomaly": any(row["interval_error"] is not None for row in rows),
        }
    )

CLI = {}
CLI_REQUEST_COVERAGE = []
for station in sorted(CONFIG_BY_STATION):
    event_days = [row["climate_date"] for row in EVENT_ROWS if row["station"] == station]
    assert event_days, f"No settled history for configured station {station}"
    for year in range(min(event_days).year, max(event_days).year + 1):
        payload, provenance = get_json(
            "https://mesonet.agron.iastate.edu/json/cli.py",
            {
                "station": station,
                "year": year,
            },
        )
        CLI_REQUEST_COVERAGE.append(
            {
                "station": station,
                "year": year,
                "days": len(payload["results"]),
                "cache_path": provenance,
            }
        )
        for row in payload["results"]:
            assert row["station"] == station, row
            day = date.fromisoformat(row["valid"])
            assert day.year == year
            assert (station, day) not in CLI, (station, day)
            high = row["high"]
            assert (type(high) is int) or high == "M", (station, day, high)
            product_parts = row["product"].split("-")
            assert len(product_parts) in (4, 5), row
            assert product_parts[3] == CONFIG_BY_STATION[station]["cli_code"], row
            CLI[station, day] = {**row, "cache_path": provenance}
    print("CLI cached:", station, min(event_days), "through", max(event_days), flush=True)

MISSING_CLI = []
DISAGREEMENTS = []
for event in EVENT_ROWS:
    cli = CLI.get((event["station"], event["climate_date"]))
    event["cli_max"] = cli["high"] if cli and type(cli["high"]) is int else None
    event["cli_text"] = ("https://mesonet.agron.iastate.edu" + cli["link"]) if cli else None
    event["cli_product"] = cli["product"] if cli else None
    event["cli_missing_reason"] = (
        "absent row" if cli is None else ("reported M" if cli["high"] == "M" else None)
    )
    if event["cli_missing_reason"]:
        MISSING_CLI.append(
            {
                key: event[key]
                for key in [
                    "series",
                    "event_ticker",
                    "station",
                    "climate_date",
                    "cli_missing_reason",
                    "cli_text",
                ]
            }
        )
    # Every uncomparable event remains in EVENT_ROWS and the category counts.
    event["comparable"] = (
        event["yes_count"] == 1 and event["cli_max"] is not None and not event["date_anomaly"]
    )
    event["disagreement"] = (
        not contains(Decimal(event["cli_max"]), event["yes_interval"])
        if event["comparable"]
        else None
    )
    event["exact_difference_f"] = (
        event["expiration_value"] - Decimal(event["cli_max"])
        if event["expiration_value"] is not None
        and event["cli_max"] is not None
        and not event["date_anomaly"]
        else None
    )
    if event["disagreement"]:
        DISAGREEMENTS.append(event)

SERIES_SUMMARY = []
for series in sorted({row["series"] for row in NORMALIZED}):
    markets = [row for row in NORMALIZED if row["series"] == series]
    events = [row for row in EVENT_ROWS if row["series"] == series]
    comparable = [row for row in events if row["comparable"]]
    twc = [row for row in markets if row["source"] == "The Weather Company"]
    first_twc = min((row["climate_date"] for row in twc), default=None)
    dates = sorted({row["climate_date"] for row in events})
    mismatches = sum(row["disagreement"] is True for row in comparable)
    SERIES_SUMMARY.append(
        {
            "series": series,
            "station": markets[0]["station"],
            "family": markets[0]["family"],
            "markets": len(markets),
            "events": len(events),
            "first_date": dates[0],
            "last_date": dates[-1],
            "first_TWC_date": first_twc,
            "comparable_events": len(comparable),
            "disagreements": mismatches,
            "disagreement_rate_pct": round(100 * mismatches / len(comparable), 4)
            if comparable
            else None,
            "missing_CLI_events": sum(row["cli_missing_reason"] is not None for row in events),
            "zero_or_multiple_YES": sum(row["yes_count"] != 1 for row in events),
            "date_anomaly_events": sum(row["date_anomaly"] for row in events),
            "numeric_expiration_markets": sum(
                row["expiration_value"] is not None for row in markets
            ),
            "blank_expiration_markets": sum(row["expiration_raw"] in ("", None) for row in markets),
            "nonnumeric_expiration_markets": sum(
                row["expiration_error"] is not None for row in markets
            ),
        }
    )

EXACT_DIFFERENCES = []
for series in sorted({row["series"] for row in NORMALIZED}):
    events = [row for row in EVENT_ROWS if row["series"] == series]
    counts = Counter(
        row["exact_difference_f"] for row in events if row["exact_difference_f"] is not None
    )
    market_counts = Counter()
    for row in NORMALIZED:
        if row["series"] != series or row["expiration_value"] is None or row["date_error"]:
            continue
        cli = CLI.get((row["station"], row["climate_date"]))
        if cli and type(cli["high"]) is int:
            market_counts[row["expiration_value"] - Decimal(cli["high"])] += 1
    for difference in sorted(counts.keys() | market_counts.keys()):
        EXACT_DIFFERENCES.append(
            {
                "series": series,
                "expiration_minus_CLI_F": str(difference),
                "events": counts[difference],
                "market_records": market_counts[difference],
            }
        )

GAPS = []
for summary in SERIES_SUMMARY:
    days = {row["climate_date"] for row in EVENT_ROWS if row["series"] == summary["series"]}
    expected = pd.date_range(summary["first_date"], summary["last_date"], freq="D")
    for timestamp in expected:
        if timestamp.date() not in days:
            GAPS.append(
                {
                    "series": summary["series"],
                    "climate_date": timestamp.date(),
                    "classification": "No settled event served in /events inventory",
                }
            )

print("Per-series results:")
show_table(
    pd.DataFrame(SERIES_SUMMARY)[
        [
            "series",
            "station",
            "markets",
            "events",
            "first_date",
            "last_date",
            "first_TWC_date",
            "comparable_events",
            "disagreements",
            "disagreement_rate_pct",
        ]
    ]
)
print("Every market expiration_value is classified, including blank and nonnumeric values:")
show_table(
    pd.DataFrame(SERIES_SUMMARY)[
        [
            "series",
            "markets",
            "numeric_expiration_markets",
            "blank_expiration_markets",
            "nonnumeric_expiration_markets",
            "missing_CLI_events",
            "zero_or_multiple_YES",
            "date_anomaly_events",
        ]
    ]
)
print("Exact differences: expiration_value minus CLI maximum, degrees Fahrenheit:")
show_table(EXACT_DIFFERENCES)
print("Every bracket-level disagreement:")
show_table(
    [
        {
            key: row[key]
            for key in [
                "yes_ticker",
                "climate_date",
                "station",
                "source",
                "cli_max",
                "yes_bracket",
                "expiration_value",
                "cli_text",
            ]
        }
        for row in DISAGREEMENTS
    ]
)
print("Missing CLI event-days:", len(MISSING_CLI))
show_table(MISSING_CLI)
print("Event anomalies:")
show_table(EVENT_ANOMALIES)
print("Market anomalies:")
show_table(MARKET_ANOMALIES)
print("Missing settled-event dates inside each served series range:", len(GAPS))
show_table(GAPS)

CLI cached: KAUS 2023-05-11 through 2026-11-11


CLI cached: KDEN 2024-11-20 through 2026-11-11


CLI cached: KLAX 2025-01-05 through 2026-11-11


CLI cached: KMDW 2021-08-19 through 2026-11-11


CLI cached: KMIA 2023-05-11 through 2026-11-11


CLI cached: KNYC 2021-07-17 through 2026-11-11


... (13 outputs left out)

Market anomalies:


ticker,climate_date,category,detail,expiration_raw
HIGHCHI-21OCT01-T84,2021-10-01,expiration outside YES bracket,84 outside x > 84°F,NaN
HIGHNY0-21JUL17-T90,2021-07-17,date_error,Unrecognized event date: HIGHNY0-21JUL1,91
KXHIGHLAX-26OCT22-T71,2026-10-22,expiration_error,Not a numeric value: '70° to 71°',70° to 71°
KXHIGHLAX-26OCT22-T64,2026-10-22,expiration_error,Not a numeric value: '70° to 71°',70° to 71°
KXHIGHLAX-26OCT22-B70.5,2026-10-22,expiration_error,Not a numeric value: '70° to 71°',70° to 71°
KXHIGHLAX-26OCT22-B68.5,2026-10-22,expiration_error,Not a numeric value: '70° to 71°',70° to 71°
KXHIGHLAX-26OCT22-B66.5,2026-10-22,expiration_error,Not a numeric value: '70° to 71°',70° to 71°
KXHIGHLAX-26OCT22-B64.5,2026-10-22,expiration_error,Not a numeric value: '70° to 71°',70° to 71°
KXHIGHMIA-24NOV05-B84.5,2024-11-05,expiration_error,"Not a numeric value: '84. Although the Daily Climate Report was not finalized for November 5th, the NWS provides hourly high data which indicates a high of 84 degrees on November 5, 2024. As said in previous guidance, Kalshi in these situations will seek to use data provided by the same Source Agency on the same topic when the primary reference is not available.'","84. Although the Daily Climate Report was not finalized for November 5th, the NWS provides hourly high data which indicates a high of 84 degrees on November 5, 2024. As said in previous guidance, Kalshi in these situations will seek to use data provided by the same Source Agency on the same topic when the primary reference is not available."
KXHIGHMIA-26APR11-T83,2026-04-11,expiration_error,Not a numeric value: 'No',No


Missing settled-event dates inside each served series range: 13


series,climate_date,classification
HIGHAUS,2023-08-27,No settled event served in /events inventory
HIGHAUS,2023-11-19,No settled event served in /events inventory
HIGHAUS,2023-12-24,No settled event served in /events inventory
HIGHAUS,2024-01-07,No settled event served in /events inventory
HIGHCHI,2021-09-20,No settled event served in /events inventory
HIGHCHI,2021-12-26,No settled event served in /events inventory
HIGHCHI,2021-12-27,No settled event served in /events inventory
HIGHCHI,2023-12-24,No settled event served in /events inventory
HIGHMIA,2023-12-24,No settled event served in /events inventory
HIGHNY,2021-08-19,No settled event served in /events inventory


## Original CLI evidence and correction timelines

Every bracket disagreement and every nonzero exact difference is checked against the
original CLI text. Positive controls check the same parser on agreements. All apparent
same-day latest products are also fetched to identify preliminary labels. An IEM row
that disagrees with its linked text stays in an explicit quality-anomaly table.

In [5]:
def parse_cli_text(body, station, expected_day):
    text = body.decode("utf-8")
    cli_code = CONFIG_BY_STATION[station]["cli_code"]
    assert re.search(r"^" + re.escape(cli_code) + r"\s*$", text, re.MULTILINE), cli_code
    headlines = list(re.finditer(r"CLIMATE SUMMARY FOR\s+([A-Z]+\s+\d{1,2}\s+\d{4})", text))
    sections = []
    for index, match in enumerate(headlines):
        days = text_dates(match[1])
        assert len(days) == 1, match[1]
        if next(iter(days)) == expected_day:
            end = headlines[index + 1].start() if index + 1 < len(headlines) else len(text)
            sections.append(text[match.start() : end])
    assert len(sections) == 1, (station, expected_day, "CLI climate date mismatch")
    section = sections[0]
    matches = list(re.finditer(r"^\s*MAXIMUM\s+(-?\d+|MM|M)(?:R)?\b[^\n]*", section, re.MULTILINE))
    assert len(matches) == 1, (station, expected_day, "CLI maximum missing or ambiguous")
    maximum = matches[0][1]
    return {
        "station": station,
        "climate_date": expected_day,
        "maximum": None if maximum in {"M", "MM"} else int(maximum),
        "maximum_line": matches[0][0].strip(),
        "preliminary_same_day": "VALID TODAY AS OF" in section,
        "correction": bool(re.search(r"\bCORRECT(?:ED|ION)\b|\bCC[A-Z]\b", text)),
        "text": text,
    }


# Check every bracket discrepancy and every nonzero exact difference, including
# old legacy prototypes whose thresholds are not an exhaustive bracket partition.
EVIDENCE_TARGETS = {
    row["event_ticker"]: row
    for row in EVENT_ROWS
    if row["disagreement"] or row["exact_difference_f"] not in (None, Decimal(0))
}
for anomaly in MARKET_ANOMALIES:
    if anomaly["category"] == "expiration outside YES bracket":
        market = next(row for row in NORMALIZED if row["ticker"] == anomaly["ticker"])
        event = next(row for row in EVENT_ROWS if row["event_ticker"] == market["event_ticker"])
        EVIDENCE_TARGETS[event["event_ticker"]] = event
# Positive controls exercise both historical and current primary-source wording.
for family in ["HIGH", "KX"]:
    event = next(
        row
        for row in EVENT_ROWS
        if row["family"] == family
        and row["comparable"]
        and row["source"] != "Not stated in rules_primary"
        and (family == "HIGH" or row["source"] == "The Weather Company")
    )
    EVIDENCE_TARGETS[event["event_ticker"]] = event

# A product timestamp on/before the climate date (UTC) is a candidate for an
# unfinished same-day report at these US stations. Confirm from the actual text;
# this timestamp is only a diagnostic and is never used as a join date.
for event in EVENT_ROWS:
    if event["cli_product"] and event["cli_max"] is not None:
        product_day = (
            datetime.strptime(event["cli_product"][:8], "%Y%m%d").replace(tzinfo=UTC).date()
        )
        if product_day <= event["climate_date"]:
            EVIDENCE_TARGETS[event["event_ticker"]] = event

TEXT_EVIDENCE = {}
AUDIT_PAYLOADS = {}
AUDIT_CLI_ROWS = []
CLI_TEXT_ANOMALIES = []
for event_ticker, event in sorted(EVIDENCE_TARGETS.items()):
    assert event["cli_text"], event_ticker
    body, status, provenance = cached_response(event["cli_text"])
    parsed = parse_cli_text(body, event["station"], event["climate_date"])
    if parsed["maximum"] != event["cli_max"]:
        CLI_TEXT_ANOMALIES.append(
            {
                "event_ticker": event_ticker,
                "station": event["station"],
                "climate_date": event["climate_date"],
                "iem_high": event["cli_max"],
                "linked_text_high": parsed["maximum"],
                "cli_text": event["cli_text"],
                "reason": "IEM annual high disagrees with its own linked CLI text",
            }
        )
    TEXT_EVIDENCE[event_ticker] = {
        **{key: value for key, value in parsed.items() if key != "text"},
        "event_ticker": event_ticker,
        "cli_text": event["cli_text"],
        "cache_path": provenance,
        "exact_difference_f": event["exact_difference_f"],
        "iem_high": event["cli_max"],
        "yes_bracket": event["yes_bracket"],
        "source": event["source"],
    }
    if event["disagreement"] or event["exact_difference_f"] not in (None, Decimal(0)):
        audit, audit_provenance = get_json(
            "https://mesonet.agron.iastate.edu/json/cli_audit.py",
            {
                "station": event["station"],
                "date": event["climate_date"].isoformat(),
            },
        )
        AUDIT_PAYLOADS[event_ticker] = {"payload": audit, "cache_path": audit_provenance}
        # The provider's audit also includes observations, DSM and CF6. Only CLI
        # revisions are relevant to this comparison; raw audit bodies retain all.
        assert audit["station"] == event["station"]
        assert audit["date"] == event["climate_date"].isoformat()
        audit_events = audit["high"]["events"]
        for entry in audit_events:
            if entry["source"] == "CLI" and entry["varname"] == "high":
                AUDIT_CLI_ROWS.append({"event_ticker": event_ticker, **entry})
    print("Original CLI checked:", event_ticker, parsed["maximum_line"], flush=True)

REVISION_TEXT_EVIDENCE = []
for event in DISAGREEMENTS:
    for entry in AUDIT_CLI_ROWS:
        if entry["event_ticker"] != event["event_ticker"]:
            continue
        product_id = parse_qs(urlsplit(entry["link"]).query)["pid"][0]
        link = "https://mesonet.agron.iastate.edu/api/1/nwstext/" + product_id
        body, status, provenance = cached_response(link)
        parsed = parse_cli_text(body, event["station"], event["climate_date"])
        assert Decimal(parsed["maximum"]) == Decimal(str(entry["value"]))
        REVISION_TEXT_EVIDENCE.append(
            {
                "event_ticker": event["event_ticker"],
                "issued_at_UTC": entry["utc_valid"],
                "cli_max": parsed["maximum"],
                "maximum_line": parsed["maximum_line"],
                "preliminary_same_day": parsed["preliminary_same_day"],
                "cli_text": link,
                "cache_path": provenance,
            }
        )

print("Original CLI spot-checks (every bracket discrepancy and every exact discrepancy):")
show_table(list(TEXT_EVIDENCE.values()))
print("CLI issuance/correction history for discrepancy investigations:")
show_table(AUDIT_CLI_ROWS)
print("Original text behind both bracket-disagreement revision timelines:")
show_table(REVISION_TEXT_EVIDENCE)
print("Latest IEM labels confirmed to be preliminary same-day products:")
show_table([row for row in TEXT_EVIDENCE.values() if row["preliminary_same_day"]])
print("IEM high versus linked-text inconsistencies (retained, never silently corrected):")
show_table(CLI_TEXT_ANOMALIES)

Original CLI checked: HIGHAUS-23AUG01 MAXIMUM        103   2:53 PM 105    2011  98      5      100


Original CLI checked: HIGHCHI-21OCT01 MAXIMUM         86   3:15 PM  92    1971  70     16       58


Original CLI checked: HIGHCHI-22APR02 MAXIMUM         45  12:04 PM  82    1963  55    -10       51


Original CLI checked: HIGHCHI-23JAN04 MAXIMUM         45  12:01 AM  64    1997  33     12       36


Original CLI checked: HIGHNY0-21AUG02 MAXIMUM         80    347 PM 100    1955  85     -5       89


Original CLI checked: HIGHNY0-21JUL19 MAXIMUM         81    447 PM 102    1977  85     -4       94


... (24 outputs left out)

Latest IEM labels confirmed to be preliminary same-day products:


station,climate_date,maximum,maximum_line,preliminary_same_day,correction,event_ticker,cli_text,cache_path,exact_difference_f,iem_high,yes_bracket,source
KAUS,2026-02-21,74,MAXIMUM 74 2:15 PM 101 1996 68 6 38,True,False,KXHIGHAUS-26FEB21,https://mesonet.agron.iastate.edu/api/1/nwstext/202602212355-KEWX-CDUS44-CLIAUS,data/spike/settlement_source/iem/2026-11-12/204da25687326bba5df6caf1c344f8e07583f81b602998a30b36803e4b21ffac.json.gz,0.00,74,"[73, 74]°F",NWS CLI
KDEN,2026-02-21,46,MAXIMUM 46 304 PM 73 2017 47 -1 41,True,False,KXHIGHDEN-26FEB21,https://mesonet.agron.iastate.edu/api/1/nwstext/202602212330-KBOU-CDUS45-CLIDEN,data/spike/settlement_source/iem/2026-11-12/da292efde3c162621d54c5a64297ab06ad26cd04c131643cded3d2c6be7985e7.json.gz,0.00,46,"[45, 46]°F",NWS CLI
KMIA,2025-05-07,86,MAXIMUM 86 1:58 PM 93 2021 86 0 86,True,False,KXHIGHMIA-25MAY07,https://mesonet.agron.iastate.edu/api/1/nwstext/202505072023-KMFL-CDUS42-CLIMIA,data/spike/settlement_source/iem/2026-11-12/29fca31c1fb7912aef7f9dd442536adf3ee2d90010c47d065e6b5b673c98b223.json.gz,1.00,86,"[86, 87]°F",NWS CLI
KMIA,2026-02-21,84,MAXIMUM 84 2:51 PM 86 1989 79 5 75,True,False,KXHIGHMIA-26FEB21,https://mesonet.agron.iastate.edu/api/1/nwstext/202602212123-KMFL-CDUS42-CLIMIA,data/spike/settlement_source/iem/2026-11-12/6c1503368647d203e1f290be8afcf7d9c24a82754d75d46b3c29b82d4cacc747.json.gz,0.00,84,"[83, 84]°F",NWS CLI
KNYC,2025-06-18,83,MAXIMUM 83 259 PM 95 1929 81 2 86,True,False,KXHIGHNY-25JUN18,https://mesonet.agron.iastate.edu/api/1/nwstext/202506182032-KOKX-CDUS41-CLINYC,data/spike/settlement_source/iem/2026-11-12/9b78ed1505be7ca0a0d82704b97a881963649a1f8ca0592299c950d7bc7b1c58.json.gz,1.00,83,"[83, 84]°F",NWS CLI
KNYC,2025-11-13,51,MAXIMUM 51 307 PM 73 1931 55 -4 50,True,False,KXHIGHNY-25NOV13,https://mesonet.agron.iastate.edu/api/1/nwstext/202511132136-KOKX-CDUS41-CLINYC,data/spike/settlement_source/iem/2026-11-12/1027ba080c8e27eaef2987cf24145cdf198190a8618082d34d3e330aa7937ff8.json.gz,0.00,51,"[50, 51]°F",NWS CLI
KPHL,2025-01-02,43,MAXIMUM 43 1:13 PM 67 1876 42 1 44,True,False,KXHIGHPHIL-25JAN02,https://mesonet.agron.iastate.edu/api/1/nwstext/202501022149-KPHI-CDUS41-CLIPHL,data/spike/settlement_source/iem/2026-11-12/2904b0d0599bb3e82877ae9670444bd34b264f84b72fb95475a9b6a8fe555b99.json.gz,0.00,43,"[42, 43]°F",NWS CLI
KPHL,2026-08-18,87,MAXIMUM 87 MM 97 1995 86 1 MM,True,True,KXHIGHPHIL-26AUG18,https://mesonet.agron.iastate.edu/api/1/nwstext/202608182200-KPHI-CDUS41-CLIPHL-CCA,data/spike/settlement_source/iem/2026-11-12/da31c73c000e7834670cd4ac6be49eb3f230ca8012c88e6df7aaf8a4fee71f64.json.gz,0.00,88,"[88, 89]°F",The Weather Company


IEM high versus linked-text inconsistencies (retained, never silently corrected):


event_ticker,station,climate_date,iem_high,linked_text_high,cli_text,reason
KXHIGHPHIL-26AUG18,KPHL,2026-08-18,88,87,https://mesonet.agron.iastate.edu/api/1/nwstext/202608182200-KPHI-CDUS41-CLIPHL-CCA,IEM annual high disagrees with its own linked CLI text


## Preliminary and missing label investigations

For every missing label and every confirmed preliminary linked report, search four UTC
days of the appropriate CLI product and identify completed numeric reports by their
headline and absence of `VALID TODAY AS OF`. This is a bounded archive investigation,
not a guarantee that no later correction exists. The main baseline preserves the exact
IEM annual response; the table shows whether a completed report can corroborate or
qualify it. No values are silently repaired.

In [6]:
QUALITY_TARGETS = {
    row["event_ticker"]: row
    for row in EVENT_ROWS
    if row["cli_missing_reason"]
    or (
        row["event_ticker"] in TEXT_EVIDENCE
        and TEXT_EVIDENCE[row["event_ticker"]]["preliminary_same_day"]
    )
}
QUALITY_REVIEW = []
for event_ticker, event in sorted(QUALITY_TARGETS.items()):
    day = event["climate_date"]
    start = datetime(day.year, day.month, day.day, tzinfo=UTC)
    payload, provenance = get_json(
        "https://mesonet.agron.iastate.edu/json/nwstext_search.py",
        {
            "awipsid": event["cli_code"],
            "sts": start.isoformat(),
            "ets": (start + timedelta(days=4)).isoformat(),
        },
    )
    completed = []
    for product in payload["results"]:
        text = product["data"]
        headlines = re.findall(r"CLIMATE SUMMARY FOR\s+([A-Z]+\s+\d{1,2}\s+\d{4})", text)
        if not any(text_dates(headline) == {day} for headline in headlines):
            continue  # Other climate days in this explicitly bounded text search.
        parsed = parse_cli_text(text.encode(), event["station"], day)
        if not parsed["preliminary_same_day"] and parsed["maximum"] is not None:
            completed.append(
                {
                    "issued_at_UTC": product["utcvalid"],
                    "maximum": parsed["maximum"],
                    "maximum_line": parsed["maximum_line"],
                }
            )
    completed.sort(key=lambda row: row["issued_at_UTC"])
    latest = completed[-1] if completed else None
    QUALITY_REVIEW.append(
        {
            "event_ticker": event_ticker,
            "station": event["station"],
            "climate_date": day,
            "stored_iem_high": event["cli_max"],
            "stored_cli_text": event["cli_text"],
            "completed_cli_count_in_four_day_search": len(completed),
            "latest_completed_cli_high": latest["maximum"] if latest else None,
            "latest_completed_issued_at_UTC": latest["issued_at_UTC"] if latest else None,
            "search_cache_path": provenance,
            "conclusion": (
                "No completed numeric CLI found in this bounded archive search"
                if latest is None
                else "Completed CLI available in text archive"
            ),
        }
    )
    print("Label quality checked:", event_ticker, QUALITY_REVIEW[-1]["conclusion"], flush=True)
print("Preliminary and missing-label investigations; baseline values remain verbatim:")
show_table(QUALITY_REVIEW)

Label quality checked: HIGHMIA-23OCT10 No completed numeric CLI found in this bounded archive search


Label quality checked: KXHIGHAUS-26FEB21 Completed CLI available in text archive


Label quality checked: KXHIGHDEN-26FEB21 Completed CLI available in text archive


Label quality checked: KXHIGHLAX-25JUN10 No completed numeric CLI found in this bounded archive search


Label quality checked: KXHIGHLAX-25JUN11 No completed numeric CLI found in this bounded archive search


Label quality checked: KXHIGHMIA-24NOV05 No completed numeric CLI found in this bounded archive search


... (12 outputs left out)

Label quality checked: KXHIGHPHIL-25JAN02 No completed numeric CLI found in this bounded archive search


Label quality checked: KXHIGHPHIL-26AUG18 Completed CLI available in text archive


Preliminary and missing-label investigations; baseline values remain verbatim:


event_ticker,station,climate_date,stored_iem_high,stored_cli_text,completed_cli_count_in_four_day_search,latest_completed_cli_high,latest_completed_issued_at_UTC,search_cache_path,conclusion
HIGHMIA-23OCT10,KMIA,2023-10-10,NaN,NaN,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/58e2901707a48c90d86570535fa74a80c4ee4ff7ca7d452de7956e9261e53e88.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHAUS-26FEB21,KAUS,2026-02-21,74.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202602212355-KEWX-CDUS44-CLIAUS,3,74.0,2026-02-22T08:18Z,data/spike/settlement_source/iem/2026-11-12/0da0f3272349c0a1595a27ed7844ab3ccb1ff9f1612c33c44597d0191dc36f4e.json.gz,Completed CLI available in text archive
KXHIGHDEN-26FEB21,KDEN,2026-02-21,46.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202602212330-KBOU-CDUS45-CLIDEN,3,46.0,2026-02-22T07:38Z,data/spike/settlement_source/iem/2026-11-12/6c73d83afcd2b71eed7e5970653fae4b62a9ab8ea959f4ce82024c49662bee63.json.gz,Completed CLI available in text archive
KXHIGHLAX-25JUN10,KLAX,2025-06-10,NaN,NaN,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/765f7c068bf5ad61a7768839683a7f5dec7ccd66d0160150ff019ef20dbf07ae.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHLAX-25JUN11,KLAX,2025-06-11,NaN,NaN,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/698a085c2c1a43d7d43151798005e9c3012bf9c6fa8a5a722b7c4137c0f3960c.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHMIA-24NOV05,KMIA,2024-11-05,NaN,https://mesonet.agron.iastate.edu/api/1/nwstext/202411060926-KMFL-CDUS42-CLIMIA,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/148fed976813e9656fa6a99c7791b3cf47419526fc940313b46e51819991c422.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHMIA-25APR29,KMIA,2025-04-29,NaN,NaN,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/f32e24c38cc6cb3bc4a7a29bbec1e8cb4d66c40811fdbdccbd33caa9fbb463d9.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHMIA-25APR30,KMIA,2025-04-30,NaN,NaN,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/540ef7ccacb647cc16e9140a2d1af8bbe4e651c394caeb7e177b8cc4c5fbaaa5.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHMIA-25JUN01,KMIA,2025-06-01,NaN,https://mesonet.agron.iastate.edu/api/1/nwstext/202506020824-KMFL-CDUS42-CLIMIA,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/a4f22157f13aa1e8992b4373957f41997a93121d58faa8ca2a3f39f25db66998.json.gz,No completed numeric CLI found in this bounded archive search
KXHIGHMIA-25JUN02,KMIA,2025-06-02,NaN,https://mesonet.agron.iastate.edu/api/1/nwstext/202506030825-KMFL-CDUS42-CLIMIA,0,NaN,NaN,data/spike/settlement_source/iem/2026-11-12/acd866003a33e2bc24a322bec4dca46051fccf07ca1697268de47bacd9d0dcbe.json.gz,No completed numeric CLI found in this bounded archive search


## Required-property checks

Each full-sentence check validates either a data invariant or complete exception reporting.
Real-data failures such as zero/multiple YES outcomes and the out-of-bracket expiration
remain in the anomaly tables; successful reporting checks do not claim those source
records are clean.

In [7]:
def market_records_for_validation():
    return MARKETS


def yes_anomalies_for_validation():
    return EVENT_ANOMALIES


def missing_cli_for_validation():
    return MISSING_CLI


def evidence_for_validation():
    return TEXT_EVIDENCE


def expiration_anomalies_for_validation():
    return [row for row in MARKET_ANOMALIES if row["category"] == "expiration outside YES bracket"]


def legacy_rows_for_validation():
    return [row for row in EVENT_ROWS if row["family"] == "HIGH"]


def exact_differences_for_validation():
    return EXACT_DIFFERENCES


def check_every_inventoried_market_is_accounted_for_exactly_once():
    """Every inventoried market is accounted for exactly once across the cutoff."""
    records = market_records_for_validation()
    tickers = [row["ticker"] for row in records]
    assert len(tickers) == len(set(tickers)), "Duplicate market ticker in final history"
    expected = {
        ticker for page in PAGES if page["path"] != "/events" for ticker in page["tickers"]
    } - set(AFTER_SNAPSHOT)
    assert set(tickers) == expected, "Market page was lost or an unfetched ticker was added"
    actual_events = {row["event_ticker"] for row in records}
    assert actual_events == set(EVENTS), (
        "Independent /events inventory differs",
        sorted(set(EVENTS) - actual_events),
        sorted(actual_events - set(EVENTS)),
    )
    assert len(NORMALIZED) == len(records), "A boundary record was silently dropped"
    groups = defaultdict(list)
    for page in PAGES:
        groups[page["path"], page["series"]].append(page)
    for key, pages in groups.items():
        assert pages[0]["cursor_in"] == "" and pages[-1]["cursor_out"] == "", key
        assert all(
            a["cursor_out"] == b["cursor_in"] for a, b in zip(pages, pages[1:], strict=False)
        ), key
        incoming = [page["cursor_in"] for page in pages]
        assert len(incoming) == len(set(incoming)), ("Repeated page", key)
        ids = [ticker for page in pages for ticker in page["tickers"]]
        assert len(ids) == len(set(ids)), ("Repeated record within one endpoint", key)
    assert CUTOFF_START["market_settled_ts"] == CUTOFF_END["market_settled_ts"]
    assert sum(row["markets"] for row in SERIES_SUMMARY) == len(records)


def check_every_nonunique_yes_event_is_listed():
    """Every event has one YES bracket or is explicitly listed with its actual count."""
    actual = Counter(row["event_ticker"] for row in NORMALIZED if row["result"] == "yes")
    expected = {ticker: actual[ticker] for ticker in EVENTS if actual[ticker] != 1}
    reported = {
        row["event_ticker"]: row["yes_count"]
        for row in yes_anomalies_for_validation()
        if "yes_count" in row
    }
    assert expected == reported, "A zero-YES or multiple-YES event was omitted or miscounted"
    assert all(row["yes_count"] == actual[row["event_ticker"]] for row in EVENT_ROWS)
    assert {row["event_ticker"] for row in EVENT_ROWS} == set(EVENTS)


RAW_BY_TICKER = {row["ticker"]: row for row in MARKETS}


def check_every_join_uses_config_and_the_rule_climate_date():
    """Every join uses config station identity and the market's cross-checked climate date."""
    for row in NORMALIZED:
        raw = RAW_BY_TICKER[row["ticker"]]
        event = EVENTS[row["event_ticker"]]
        station, _ = station_from_evidence(raw, event)
        assert (
            row["station"] == station and row["cli_code"] == CONFIG_BY_STATION[station]["cli_code"]
        )
        try:
            verified_day = climate_date(raw, event)
        except AssertionError:
            assert row["date_error"], f"Unreported date conflict: {row['ticker']}"
            assert any(
                a["ticker"] == row["ticker"] and a["category"] == "date_error"
                for a in MARKET_ANOMALIES
            )
            assert text_dates(raw["rules_primary"]) == {row["climate_date"]}
        else:
            assert row["climate_date"] == verified_day, (
                "Join used a timestamp date instead of climate date"
            )
        assert not (
            row["date_error"]
            and next(e for e in EVENT_ROWS if e["event_ticker"] == row["event_ticker"])[
                "comparable"
            ]
        )


def check_membership_respects_documented_and_literal_rule_bounds():
    """Bracket membership respects strict and inclusive bounds from docs and literal rules."""
    # Independent inequalities exercise equality and neighbors for every actual
    # rule interval, including old records lacking structured strike fields.
    intervals = {row["interval"] for row in NORMALIZED}
    for interval in intervals:
        lower, upper, lower_closed, upper_closed = interval
        probes = {
            point + step
            for point in (lower, upper)
            if point is not None
            for step in [Decimal(-1), Decimal(0), Decimal(1)]
        }
        for value in probes:
            expected = True
            if lower is not None:
                expected = expected and (value >= lower if lower_closed else value > lower)
            if upper is not None:
                expected = expected and (value <= upper if upper_closed else value < upper)
            assert contains(value, interval) == expected, (
                f"Wrong inclusivity at {value}: {interval}"
            )
    for row in NORMALIZED:
        assert row["interval"] == interval_from_rules(row["rules_primary"])
    for event in EVENT_ROWS:
        if event["comparable"]:
            assert event["disagreement"] == (
                not contains(Decimal(event["cli_max"]), event["yes_interval"])
            )


def check_sources_and_switch_dates_follow_verbatim_primary_rules():
    """Every source and first-TWC date follows that market's verbatim rules_primary."""
    for row in NORMALIZED:
        primary = RAW_BY_TICKER[row["ticker"]]["rules_primary"]
        assert row["rules_primary"] == primary, "Verbatim primary rules were changed"
        assert row["source"] == classify_source(primary), (
            "Source inferred from ticker or series metadata"
        )
    for summary in SERIES_SUMMARY:
        days = [
            row["climate_date"]
            for row in NORMALIZED
            if row["series"] == summary["series"]
            and "the weather company" in row["rules_primary"].casefold()
        ]
        assert summary["first_TWC_date"] == min(days, default=None), "Wrong first-TWC climate date"


def check_legacy_controls_are_matched_and_investigated_before_kx():
    """Legacy controls are fully accounted for and any discrepancy has original CLI evidence."""
    legacy = legacy_rows_for_validation()
    expected = {row["event_ticker"] for row in EVENT_ROWS if row["family"] == "HIGH"}
    assert {row["event_ticker"] for row in legacy} == expected, "Legacy control events were omitted"
    for row in legacy:
        cli = CLI.get((row["station"], row["climate_date"]))
        maximum = cli["high"] if cli and type(cli["high"]) is int else None
        assert row["cli_max"] == maximum, "Legacy control join is wrong; stop KX interpretation"
        if row["disagreement"] or row["exact_difference_f"] not in (None, Decimal(0)):
            assert row["event_ticker"] in TEXT_EVIDENCE, "Uninvestigated legacy discrepancy"
            assert row["event_ticker"] in AUDIT_PAYLOADS, (
                "Missing legacy CLI revision investigation"
            )
    assert len(legacy) == sum(row["events"] for row in SERIES_SUMMARY if row["family"] == "HIGH")


def check_every_missing_cli_day_is_counted_and_listed():
    """Every missing CLI day is counted and listed without silently shrinking the cohort."""
    expected = {}
    for event in EVENT_ROWS:
        cli = CLI.get((event["station"], event["climate_date"]))
        reason = "absent row" if cli is None else ("reported M" if cli["high"] == "M" else None)
        if reason:
            expected[event["event_ticker"]] = reason
    actual = {
        row["event_ticker"]: row["cli_missing_reason"] for row in missing_cli_for_validation()
    }
    assert actual == expected, "A missing CLI day was silently skipped or misclassified"
    assert len(EVENT_ROWS) == len(EVENTS), "Events disappeared during CLI join"
    for summary in SERIES_SUMMARY:
        assert summary["missing_CLI_events"] == sum(
            row["series"] == summary["series"] for row in MISSING_CLI
        )


def check_required_spot_checks_match_original_cli_text():
    """Every KX bracket disagreement is verified against the original CLI text product."""
    evidence = evidence_for_validation()
    expected = {row["event_ticker"] for row in DISAGREEMENTS if row["family"] == "KX"}
    assert expected <= evidence.keys(), "A KX disagreement lacks an original CLI spot-check"
    for event_ticker in expected:
        item = evidence[event_ticker]
        event = next(row for row in EVENT_ROWS if row["event_ticker"] == event_ticker)
        assert item["cli_text"] == event["cli_text"], (
            "Evidence link does not identify the joined product"
        )
        body, _, _ = cached_response(item["cli_text"])
        parsed = parse_cli_text(body, event["station"], event["climate_date"])
        assert parsed["maximum"] == item["maximum"] == event["cli_max"], (
            "Wrong original CLI maximum"
        )
        assert item["maximum_line"] == parsed["maximum_line"]


def check_every_numeric_expiration_fits_the_yes_bracket_or_is_listed():
    """Every numeric expiration fits its event's YES bracket or appears in the anomaly report."""
    events = {row["event_ticker"]: row for row in EVENT_ROWS}
    expected = set()
    for market in NORMALIZED:
        event = events[market["event_ticker"]]
        if (
            market["expiration_value"] is not None
            and event["yes_count"] == 1
            and not contains(market["expiration_value"], event["yes_interval"])
        ):
            expected.add(market["ticker"])
    reported = {row["ticker"] for row in expiration_anomalies_for_validation()}
    assert expected == reported, "An expiration outside its YES bracket was hidden"
    for summary in SERIES_SUMMARY:
        assert summary["markets"] == sum(
            summary[key]
            for key in [
                "numeric_expiration_markets",
                "blank_expiration_markets",
                "nonnumeric_expiration_markets",
            ]
        ), "Expiration coverage does not account for every market"


def check_exact_difference_tables_account_for_all_numeric_values():
    """Exact-difference tables account for every numeric market and identifiable event maximum."""
    expected_markets = Counter()
    for row in NORMALIZED:
        cli = CLI.get((row["station"], row["climate_date"]))
        if (
            row["expiration_value"] is not None
            and not row["date_error"]
            and cli
            and type(cli["high"]) is int
        ):
            expected_markets[row["series"], row["expiration_value"] - Decimal(cli["high"])] += 1
    expected_events = Counter(
        (row["series"], row["exact_difference_f"])
        for row in EVENT_ROWS
        if row["exact_difference_f"] is not None
    )
    actual_markets = Counter(
        {
            (row["series"], Decimal(row["expiration_minus_CLI_F"])): row["market_records"]
            for row in exact_differences_for_validation()
        }
    )
    actual_events = Counter(
        {
            (row["series"], Decimal(row["expiration_minus_CLI_F"])): row["events"]
            for row in exact_differences_for_validation()
        }
    )
    assert actual_markets == expected_markets, "Exact market difference distribution lost records"
    assert actual_events == expected_events, "Exact event difference distribution lost records"


CHECKS = [
    check_every_inventoried_market_is_accounted_for_exactly_once,
    check_every_nonunique_yes_event_is_listed,
    check_every_join_uses_config_and_the_rule_climate_date,
    check_membership_respects_documented_and_literal_rule_bounds,
    check_sources_and_switch_dates_follow_verbatim_primary_rules,
    check_legacy_controls_are_matched_and_investigated_before_kx,
    check_every_missing_cli_day_is_counted_and_listed,
    check_required_spot_checks_match_original_cli_text,
    check_every_numeric_expiration_fits_the_yes_bracket_or_is_listed,
    check_exact_difference_tables_account_for_all_numeric_values,
]
for index, check in enumerate(CHECKS, start=1):
    check()
    print(f"PASS {index}: {check.__doc__}")


def prove_failure(number, function_name, replacement, check, violation):
    """Temporarily replace production behavior, run unchanged check, restore it."""
    original = globals()[function_name]
    check()
    try:
        globals()[function_name] = replacement
        try:
            check()
        except AssertionError as error:
            message = str(error)
            print(f"EXPECTED FAILURE {number}: {violation}\nAssertionError: {message}")
        else:
            raise RuntimeError(f"Mutation {number} survived: {violation}")
    finally:
        globals()[function_name] = original
    check()
    print(f"RESTORED PASS {number}: {check.__doc__}")
    return {
        "property": number,
        "check": check.__name__,
        "violation": violation,
        "observed_failure": "AssertionError: " + message,
        "restored": "PASS",
    }


PROOFS = []

PASS 1: Every inventoried market is accounted for exactly once across the cutoff.
PASS 2: Every event has one YES bracket or is explicitly listed with its actual count.


PASS 3: Every join uses config station identity and the market's cross-checked climate date.
PASS 4: Bracket membership respects strict and inclusive bounds from docs and literal rules.


PASS 5: Every source and first-TWC date follows that market's verbatim rules_primary.
PASS 6: Legacy controls are fully accounted for and any discrepancy has original CLI evidence.
PASS 7: Every missing CLI day is counted and listed without silently shrinking the cohort.
PASS 8: Every KX bracket disagreement is verified against the original CLI text product.
PASS 9: Every numeric expiration fits its event's YES bracket or appears in the anomaly report.
PASS 10: Exact-difference tables account for every numeric market and identifiable event maximum.


## Proof that each check fires

Each demonstration temporarily replaces a production loader, parser, comparator or report
path with broken behavior, runs the unchanged check, prints its `AssertionError`, restores
the original function in `finally`, and reruns successfully. All examples use real cached
records. The notebook finishes with the original behavior restored.

In [8]:
PROOFS.append(
    prove_failure(
        "1a",
        "market_records_for_validation",
        lambda: MARKETS[1:],
        CHECKS[0],
        "The loader drops a real cached settled market.",
    )
)
PROOFS.append(
    prove_failure(
        "1b",
        "market_records_for_validation",
        lambda: MARKETS + [MARKETS[0]],
        CHECKS[0],
        "The loader emits the same ticker twice across the cutoff.",
    )
)
for count in [0, 2]:
    omitted = next(row["event_ticker"] for row in EVENT_ANOMALIES if row.get("yes_count") == count)
    PROOFS.append(
        prove_failure(
            f"2 ({count} YES)",
            "yes_anomalies_for_validation",
            lambda omitted=omitted: [
                row for row in EVENT_ANOMALIES if row["event_ticker"] != omitted
            ],
            CHECKS[1],
            f"The reporter hides a real {count}-YES event: {omitted}.",
        )
    )
PROOFS.append(
    prove_failure(
        "3",
        "climate_date",
        lambda market, event: datetime.fromisoformat(
            market["close_time"].replace("Z", "+00:00")
        ).date(),
        CHECKS[2],
        "The date parser incorrectly uses the UTC close date.",
    )
)
real_contains = contains
PROOFS.append(
    prove_failure(
        "4",
        "contains",
        lambda value, interval: real_contains(
            value, (interval[0], interval[1], interval[0] is not None, interval[1] is not None)
        ),
        CHECKS[3],
        "The interval comparator incorrectly makes strict tails inclusive.",
    )
)
PROOFS.append(
    prove_failure(
        "5",
        "classify_source",
        lambda primary: "The Weather Company",
        CHECKS[4],
        "The source classifier labels all old rules as TWC.",
    )
)
original_legacy_rows = legacy_rows_for_validation


def wrong_legacy_match():
    rows = [dict(row) for row in original_legacy_rows()]
    row = next(row for row in rows if row["comparable"])
    row["cli_max"] += 1
    return rows


PROOFS.append(
    prove_failure(
        "6",
        "legacy_rows_for_validation",
        wrong_legacy_match,
        CHECKS[5],
        "The legacy join returns a maximum from the wrong station/day.",
    )
)
PROOFS.append(
    prove_failure(
        "7",
        "missing_cli_for_validation",
        lambda: [],
        CHECKS[6],
        "The reporter silently skips every missing CLI label.",
    )
)
PROOFS.append(
    prove_failure(
        "8",
        "evidence_for_validation",
        lambda: {},
        CHECKS[7],
        "The spot-check reporter drops the original CLI evidence.",
    )
)
PROOFS.append(
    prove_failure(
        "9",
        "expiration_anomalies_for_validation",
        lambda: [],
        CHECKS[8],
        "The reporter hides a real expiration outside its YES bracket.",
    )
)
PROOFS.append(
    prove_failure(
        "10",
        "exact_differences_for_validation",
        lambda: [row for row in EXACT_DIFFERENCES if Decimal(row["expiration_minus_CLI_F"]) == 0],
        CHECKS[9],
        "The distribution code omits all nonzero exact differences.",
    )
)
show_table(PROOFS)

EXPECTED FAILURE 1a: The loader drops a real cached settled market.
AssertionError: Market page was lost or an unfetched ticker was added
RESTORED PASS 1a: Every inventoried market is accounted for exactly once across the cutoff.
EXPECTED FAILURE 1b: The loader emits the same ticker twice across the cutoff.
AssertionError: Duplicate market ticker in final history
RESTORED PASS 1b: Every inventoried market is accounted for exactly once across the cutoff.
EXPECTED FAILURE 2 (0 YES): The reporter hides a real 0-YES event: HIGHCHI-21AUG23.
AssertionError: A zero-YES or multiple-YES event was omitted or miscounted
RESTORED PASS 2 (0 YES): Every event has one YES bracket or is explicitly listed with its actual count.
EXPECTED FAILURE 2 (2 YES): The reporter hides a real 2-YES event: HIGHCHI-21DEC02.
AssertionError: A zero-YES or multiple-YES event was omitted or miscounted
RESTORED PASS 2 (2 YES): Every event has one YES bracket or is explicitly listed with its actual count.


EXPECTED FAILURE 3: The date parser incorrectly uses the UTC close date.
AssertionError: Join used a timestamp date instead of climate date


RESTORED PASS 3: Every join uses config station identity and the market's cross-checked climate date.
EXPECTED FAILURE 4: The interval comparator incorrectly makes strict tails inclusive.
AssertionError: Wrong inclusivity at 31: (Decimal('31'), None, False, False)
RESTORED PASS 4: Bracket membership respects strict and inclusive bounds from docs and literal rules.


EXPECTED FAILURE 5: The source classifier labels all old rules as TWC.
AssertionError: Source inferred from ticker or series metadata
RESTORED PASS 5: Every source and first-TWC date follows that market's verbatim rules_primary.
EXPECTED FAILURE 6: The legacy join returns a maximum from the wrong station/day.
AssertionError: Legacy control join is wrong; stop KX interpretation
RESTORED PASS 6: Legacy controls are fully accounted for and any discrepancy has original CLI evidence.
EXPECTED FAILURE 7: The reporter silently skips every missing CLI label.
AssertionError: A missing CLI day was silently skipped or misclassified
RESTORED PASS 7: Every missing CLI day is counted and listed without silently shrinking the cohort.
EXPECTED FAILURE 8: The spot-check reporter drops the original CLI evidence.
AssertionError: A KX disagreement lacks an original CLI spot-check
RESTORED PASS 8: Every KX bracket disagreement is verified against the original CLI text product.
EXPECTED FAILURE 9: The repor

EXPECTED FAILURE 10: The distribution code omits all nonzero exact differences.
AssertionError: Exact market difference distribution lost records
RESTORED PASS 10: Exact-difference tables account for every numeric market and identifiable event maximum.


property,check,violation,observed_failure,restored
1a,check_every_inventoried_market_is_accounted_for_exactly_once,The loader drops a real cached settled market.,AssertionError: Market page was lost or an unfetched ticker was added,PASS
1b,check_every_inventoried_market_is_accounted_for_exactly_once,The loader emits the same ticker twice across the cutoff.,AssertionError: Duplicate market ticker in final history,PASS
2 (0 YES),check_every_nonunique_yes_event_is_listed,The reporter hides a real 0-YES event: HIGHCHI-21AUG23.,AssertionError: A zero-YES or multiple-YES event was omitted or miscounted,PASS
2 (2 YES),check_every_nonunique_yes_event_is_listed,The reporter hides a real 2-YES event: HIGHCHI-21DEC02.,AssertionError: A zero-YES or multiple-YES event was omitted or miscounted,PASS
3,check_every_join_uses_config_and_the_rule_climate_date,The date parser incorrectly uses the UTC close date.,AssertionError: Join used a timestamp date instead of climate date,PASS
4,check_membership_respects_documented_and_literal_rule_bounds,The interval comparator incorrectly makes strict tails inclusive.,"AssertionError: Wrong inclusivity at 31: (Decimal('31'), None, False, False)",PASS
5,check_sources_and_switch_dates_follow_verbatim_primary_rules,The source classifier labels all old rules as TWC.,AssertionError: Source inferred from ticker or series metadata,PASS
6,check_legacy_controls_are_matched_and_investigated_before_kx,The legacy join returns a maximum from the wrong station/day.,AssertionError: Legacy control join is wrong; stop KX interpretation,PASS
7,check_every_missing_cli_day_is_counted_and_listed,The reporter silently skips every missing CLI label.,AssertionError: A missing CLI day was silently skipped or misclassified,PASS
8,check_required_spot_checks_match_original_cli_text,The spot-check reporter drops the original CLI evidence.,AssertionError: A KX disagreement lacks an original CLI spot-check,PASS


## Completeness, source-switch evidence and interpretation

In [9]:
FETCH_SUMMARY = []
for ticker in sorted(CANDIDATES):
    live_ids = {row["ticker"] for row in LIVE[ticker]}
    historical_ids = {row["ticker"] for row in HISTORICAL[ticker]}
    prefixes = {row["event_ticker"].rsplit("-", 1)[0] for row in LIVE[ticker] + HISTORICAL[ticker]}
    FETCH_SUMMARY.append(
        {
            "query_series": ticker,
            "live_markets": len(live_ids),
            "historical_markets": len(historical_ids),
            "overlap": len(live_ids & historical_ids),
            "unique_markets": len(live_ids | historical_ids),
            "settled_event_inventory": len(EVENT_INVENTORY[ticker]),
            "returned_history_prefixes": ", ".join(sorted(prefixes)),
        }
    )
print("Completeness across both endpoints, independently reconciled to /events:")
show_table(FETCH_SUMMARY)
print("Missing inventory events:", sorted(set(EVENTS) - GROUPED_MARKETS.keys()))
print("Market events missing from inventory:", sorted(GROUPED_MARKETS.keys() - set(EVENTS)))
print("Overlapping tickers:", len(OVERLAPS), "(all settlement-critical fields agree)")
print(
    "Non-settlement differences in overlaps:",
    Counter(key for row in OVERLAPS for key in row["non_settlement_differences"]),
)

SOURCE_COUNTS = []
for (series, source), count in sorted(
    Counter((row["series"], row["source"]) for row in NORMALIZED).items()
):
    SOURCE_COUNTS.append(
        {"series": series, "source_from_own_primary_rules": source, "market_count": count}
    )
print("Source classifications (missing primary-source wording stays explicit):")
show_table(SOURCE_COUNTS)

SWITCH_EVIDENCE = []
for summary in SERIES_SUMMARY:
    switch_day = summary["first_TWC_date"]
    if switch_day is None:
        continue
    rows = [row for row in NORMALIZED if row["series"] == summary["series"]]
    first = min(
        (row for row in rows if row["source"] == "The Weather Company"),
        key=lambda row: (
            row["climate_date"],
            RAW_BY_TICKER[row["ticker"]]["created_time"],
            row["ticker"],
        ),
    )
    earlier_nws = [
        row["climate_date"]
        for row in rows
        if row["source"] == "NWS CLI" and row["climate_date"] < switch_day
    ]
    assert all(
        row["source"] == "The Weather Company" for row in rows if row["climate_date"] >= switch_day
    ), "Source reverted after first TWC date"
    SWITCH_EVIDENCE.append(
        {
            "series": summary["series"],
            "last_NWS_climate_date": max(earlier_nws, default=None),
            "first_TWC_climate_date": switch_day,
            "first_TWC_market": first["ticker"],
            "first_TWC_market_created_at_UTC": RAW_BY_TICKER[first["ticker"]]["created_time"],
            "rules_primary_verbatim": first["rules_primary"],
            "cache_path": first["cache_path"],
        }
    )
print("Switch evidence, including each first market’s verbatim primary rules:")
show_table(SWITCH_EVIDENCE)

PHASE_SUMMARY = []
for family, source in sorted({(row["family"], row["source"]) for row in EVENT_ROWS}):
    events = [row for row in EVENT_ROWS if (row["family"], row["source"]) == (family, source)]
    comparable = [row for row in events if row["comparable"]]
    exact = [row["exact_difference_f"] for row in events if row["exact_difference_f"] is not None]
    PHASE_SUMMARY.append(
        {
            "family": family,
            "source_from_own_primary_rules": source,
            "events": len(events),
            "comparable_bracket_events": len(comparable),
            "bracket_disagreements": sum(row["disagreement"] is True for row in comparable),
            "numeric_exact_events_with_CLI": len(exact),
            "exact_difference_histogram_F": dict(sorted(Counter(map(str, exact)).items())),
        }
    )
print("KX and HIGH controls, separated again by each market’s own settlement source:")
show_table(PHASE_SUMMARY)

BRACKET_INVESTIGATIONS = []
for event in DISAGREEMENTS:
    winner = RAW_BY_TICKER[event["yes_ticker"]]
    settled_at = datetime.fromisoformat(winner["settlement_ts"].replace("Z", "+00:00"))
    timeline = [entry for entry in AUDIT_CLI_ROWS if entry["event_ticker"] == event["event_ticker"]]
    latest = max(timeline, key=lambda row: row["utc_valid"])
    latest_at = datetime.fromisoformat(latest["utc_valid"].replace("Z", "+00:00"))
    prior_match = [
        entry for entry in timeline if Decimal(str(entry["value"])) == event["expiration_value"]
    ]
    BRACKET_INVESTIGATIONS.append(
        {
            "ticker": event["yes_ticker"],
            "climate_date": event["climate_date"],
            "source_from_rules": event["source"],
            "CLI_max": event["cli_max"],
            "expiration_value": event["expiration_value"],
            "YES_bracket": event["yes_bracket"],
            "settled_at_UTC": settled_at.isoformat(),
            "latest_CLI_issued_at_UTC": latest_at.isoformat(),
            "latest_CLI_minutes_relative_to_settlement": round(
                (latest_at - settled_at).total_seconds() / 60, 2
            ),
            "earlier_CLI_also_reported_expiration_value": bool(prior_match),
            "CLI_text": event["cli_text"],
        }
    )
print("Both bracket disagreements: station/date matching and correction timing:")
show_table(BRACKET_INVESTIGATIONS)

print("Real-data exceptions retained in the results:")
print("Zero YES events:", sum(row["yes_count"] == 0 for row in EVENT_ROWS))
print("Multiple YES events:", sum(row["yes_count"] > 1 for row in EVENT_ROWS))
print("Date cross-check conflicts:", sum(row["date_error"] is not None for row in NORMALIZED))
print(
    "Intervals reconstructed from explicit rules:",
    sum(row["bounds_evidence"] == "rules_primary" for row in NORMALIZED),
)
print("Expiration outside the event YES bracket:", len(expiration_anomalies_for_validation()))
print(
    "Nonnumeric expiration values:", sum(row["expiration_error"] is not None for row in NORMALIZED)
)
print("IEM high versus linked text inconsistencies:", len(CLI_TEXT_ANOMALIES))
print(
    "Confirmed preliminary latest IEM labels:",
    sum(row["preliminary_same_day"] for row in TEXT_EVIDENCE.values()),
)
print("Full-CLI searches for missing/preliminary labels:", len(QUALITY_REVIEW))

cache_files = sorted(CACHE.rglob("*.json.gz"))
for path in cache_files:
    check = subprocess.run(
        ["git", "check-ignore", str(path.relative_to(ROOT))],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    assert check.returncode == 0, path
    envelope = json.loads(gzip.decompress(path.read_bytes()))
    body = base64.b64decode(envelope["body_base64"], validate=True)
    assert hashlib.sha256(body).hexdigest() == envelope["body_sha256"], path
print(
    f"Cache verified: {len(cache_files)} ignored exact-body responses; "
    f"{sum(path.stat().st_size for path in cache_files):,} compressed bytes."
)
print(
    f"This execution: {len(REQUESTS)} cache reads; "
    f"{sum(not row['cache'] for row in REQUESTS)} network requests."
)
if OFFLINE:
    assert all(row["cache"] for row in REQUESTS), "Offline rerun made a network request"

display(
    Markdown(
        "**Interpretation:** This is a comparison with the exact `high` values currently "
        "served by IEM, preserving missing, preliminary, corrected, and inconsistent labels. "
        "The original-text and bounded-search tables distinguish those cases. Exact numeric "
        "settlement values are available for substantial portions of both families, so "
        "bracket agreement alone is not the only evidence. The distributions show where "
        "numeric maxima differ; they do not treat blank or textual values as temperatures. "
        "CLI history can be investigated as a proxy label source for earlier days, but "
        "the measured TWC-era mismatch, revisions after settlement, preliminary IEM rows, "
        "and archive gaps rule out treating it as interchangeable settlement truth."
    )
)

Completeness across both endpoints, independently reconciled to /events:


query_series,live_markets,historical_markets,overlap,unique_markets,settled_event_inventory,returned_history_prefixes
HIGHAUS,0,0,0,0,0,
HIGHCHI,0,0,0,0,0,
HIGHDEN,0,0,0,0,0,
HIGHLAX,0,0,0,0,0,
HIGHMIA,0,0,0,0,0,
HIGHNY,0,0,0,0,0,
HIGHNY0,0,0,0,0,0,
HIGHPHIL,0,0,0,0,0,
KXDENHIGH,0,0,0,0,0,
KXHIGHAUS,402,7293,36,7659,1277,"HIGHAUS, KXHIGHAUS"


Missing inventory events: []
Market events missing from inventory: []
Overlapping tickers: 252 (all settlement-critical fields agree)
Non-settlement differences in overlaps: Counter({'open_interest_fp': 252})
Source classifications (missing primary-source wording stays explicit):


series,source_from_own_primary_rules,market_count
HIGHAUS,NWS CLI,3165
HIGHCHI,NWS CLI,5004
HIGHCHI,Not stated in rules_primary,403
HIGHCHI-2,NWS CLI,6
HIGHMIA,NWS CLI,3184
HIGHNY,NWS CLI,5110
HIGHNY,Not stated in rules_primary,306
HIGHNY0,Not stated in rules_primary,21
KXHIGHAUS,NWS CLI,3954
KXHIGHAUS,The Weather Company,540


Switch evidence, including each first market’s verbatim primary rules:


series,last_NWS_climate_date,first_TWC_climate_date,first_TWC_market,first_TWC_market_created_at_UTC,rules_primary_verbatim,cache_path
KXHIGHAUS,2026-08-13,2026-08-14,KXHIGHAUS-26AUG14-B101.5,2026-08-13T17:10:36.240667Z,"If the maximum temperature recorded at Austin for Aug 14, 2026, is between 101-102° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/dd856efc57ff9fbc6690c8a6747ebcf248e8b1e1391790c225885ed2c3defca5.json.gz
KXHIGHCHI,2026-08-13,2026-08-14,KXHIGHCHI-26AUG14-B76.5,2026-08-13T17:10:22.315427Z,"If the maximum temperature recorded at Chicago(CLIMDW) for Aug 14, 2026, is between 76-77° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/7f87006e8cb6e90be24621d8ddc0eeb5d65f164081ba2695d4d9fc27c6a8b182.json.gz
KXHIGHDEN,2026-08-13,2026-08-14,KXHIGHDEN-26AUG14-B81.5,2026-08-13T17:10:44.59521Z,"If the maximum temperature recorded at Denver for Aug 14, 2026, is between 81-82° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/c0f2d20fe06cb8afb4b54d5c7ae01d64f0313ae6bd3c0679b84158386234922f.json.gz
KXHIGHLAX,2026-08-13,2026-08-14,KXHIGHLAX-26AUG14-B71.5,2026-08-13T17:10:55.864606Z,"If the maximum temperature recorded at Los Angeles (CLILAX) for Aug 14, 2026, is between 71-72° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/1b5eae600590b9444f5db2e7fa7b91860e3e5398b0dc17d4dc10f3faa4a8dab4.json.gz
KXHIGHMIA,2026-08-13,2026-08-14,KXHIGHMIA-26AUG14-B89.5,2026-08-13T17:10:33.075536Z,"If the maximum temperature recorded at Miami for Aug 14, 2026, is between 89-90° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/f53d9d035822938005fedea6b8ca9c83878f0b037dbc960d6a6faca198122d39.json.gz
KXHIGHNY,2026-08-13,2026-08-14,KXHIGHNY-26AUG14-B85.5,2026-08-13T17:10:25.351761Z,"If the maximum temperature recorded at New York City for Aug 14, 2026, is between 85-86° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/2d0649c1a5e98c6e899b9267da5677f9adf6d9b26b01a62a609f665c35752ea9.json.gz
KXHIGHPHIL,2026-08-13,2026-08-14,KXHIGHPHIL-26AUG14-B84.5,2026-08-13T17:10:48.145749Z,"If the maximum temperature recorded at Philadelphia for Aug 14, 2026, is between 84-85° fahrenheit according to The Weather Company, then the market resolves to Yes.",data/spike/settlement_source/kalshi/2026-11-12/0d253e2b700f1000289c073e52d2ed24d0b63571c3616b711bd0da70436e3b8f.json.gz


... (3 outputs left out)

ticker,climate_date,source_from_rules,CLI_max,expiration_value,YES_bracket,settled_at_UTC,latest_CLI_issued_at_UTC,latest_CLI_minutes_relative_to_settlement,earlier_CLI_also_reported_expiration_value,CLI_text
KXHIGHMIA-24NOV27-T83,2024-11-27,NWS CLI,81,84.00,x > 83°F,2024-11-28T13:00:20.253326+00:00,2024-11-28T13:56:00+00:00,55.66,True,https://mesonet.agron.iastate.edu/api/1/nwstext/202411281356-KMFL-CDUS42-CLIMIA
KXHIGHMIA-26AUG29-B90.5,2026-08-29,The Weather Company,85,90.00,"[90, 91]°F",2026-08-30T12:00:11.155200+00:00,2026-08-30T09:10:00+00:00,-170.19,True,https://mesonet.agron.iastate.edu/api/1/nwstext/202608300910-KMFL-CDUS42-CLIMIA


Real-data exceptions retained in the results:
Zero YES events: 182
Multiple YES events: 138
Date cross-check conflicts: 1
Intervals reconstructed from explicit rules: 2971
Expiration outside the event YES bracket: 1
Nonnumeric expiration values: 11
IEM high versus linked text inconsistencies: 1
Confirmed preliminary latest IEM labels: 8
Full-CLI searches for missing/preliminary labels: 20


Cache verified: 233 ignored exact-body responses; 12,109,551 compressed bytes.
This execution: 234 cache reads; 0 network requests.


**Interpretation:** This is a comparison with the exact `high` values currently served by IEM, preserving missing, preliminary, corrected, and inconsistent labels. The original-text and bounded-search tables distinguish those cases. Exact numeric settlement values are available for substantial portions of both families, so bracket agreement alone is not the only evidence. The distributions show where numeric maxima differ; they do not treat blank or textual values as temperatures. CLI history can be investigated as a proxy label source for earlier days, but the measured TWC-era mismatch, revisions after settlement, preliminary IEM rows, and archive gaps rule out treating it as interchangeable settlement truth.

### Cache recovery provenance

During the follow-up, an inline update helper reused a notebook global and overwrote
one historical KXHIGHCHI page with notebook JSON. The updated notebook was recovered;
the original bytes of that one response were not recoverable, so exactly that public
historical page was re-fetched and its exact new body and receipt time archived.
`data/spike/settlement_source/followup_cache_recovery.json` records the affected path
and reason. The original snapshot time still governs settlement eligibility. The
source summaries, exact-difference distributions and per-series accounting were checked
against the pre-recovery results. Subsequent notebook execution is entirely offline.

## Source-specific label evidence and complete accounting

The source split uses each market's own `rules_primary`, consolidates equivalent
Decimal zero formats, and reports every TWC nonzero difference. Legacy HIGH is the
requested control family; missing primary-source wording remains explicit.

Calendar gaps are **not settled events**. The complete equation is:

`total calendar-event slots = compared + zero-YES + multiple-YES + missing CLI + calendar gap + malformed date`

`actual settled events = total calendar-event slots − calendar gaps`

Each event receives one bucket in this precedence: malformed date, missing CLI,
zero YES, multiple YES, compared. `KXHIGHMIA-26APR11` has both missing CLI and zero
YES and is counted once as missing CLI; both original diagnostic flags remain visible.
Every series sum is asserted. The aggregate is 8,538 slots = 8,193 compared + 181
exclusive zero-YES + 138 multiple-YES + 12 missing CLI + 13 gaps + 1 malformed date;
there are 8,525 actual settled events.

All 138 multiple-YES events have non-exclusive above-X predicates. For example,
`HIGHCHI-21DEC02` has >58°F and >56°F, both YES; `HIGHNY-21DEC01` has >46°F and
>44°F, both YES. Their event metadata also says `mutually_exclusive=false`.
These are nested thresholds, so satisfying the higher one also satisfies the lower
one. This explains the outcomes; comparison eligibility and rates are unchanged.

In [10]:
FOLLOWUP_REQUEST_COUNT = len(REQUESTS)
COHORT_NAMES = ("KX citing NWS CLI", "KX citing TWC", "Legacy HIGH")


def cohort_from_primary_rules(event):
    """Group KX by its actual market rules; HIGH remains the requested control family."""
    markets = GROUPED_MARKETS[event["event_ticker"]]
    sources = {classify_source(market["rules_primary"]) for market in markets}
    assert len(sources) == 1, (event["event_ticker"], sources)
    if event["family"] == "HIGH":
        return "Legacy HIGH"
    source = next(iter(sources))
    assert source in {"NWS CLI", "The Weather Company"}, (event["event_ticker"], source)
    return "KX citing TWC" if source == "The Weather Company" else "KX citing NWS CLI"


SOURCE_COHORT_SUMMARY = []
SOURCE_EXACT_DISTRIBUTION = []
for cohort in COHORT_NAMES:
    events = [event for event in EVENT_ROWS if cohort_from_primary_rules(event) == cohort]
    compared = [event for event in events if event["comparable"]]
    exact = [event for event in events if event["exact_difference_f"] is not None]
    histogram = Counter(event["exact_difference_f"] for event in exact)
    disagreements = sum(event["disagreement"] is True for event in compared)
    SOURCE_COHORT_SUMMARY.append(
        {
            "cohort": cohort,
            "total_events": len(events),
            "bracket_compared": len(compared),
            "bracket_matches": len(compared) - disagreements,
            "bracket_disagreements": disagreements,
            "bracket_agreement_pct": round(
                100 * (len(compared) - disagreements) / len(compared), 4
            ),
            "exact_compared": len(exact),
            "exact_matches": histogram[Decimal(0)],
            "exact_nonzero": len(exact) - histogram[Decimal(0)],
            "exact_agreement_pct": round(100 * histogram[Decimal(0)] / len(exact), 4),
            "exact_unavailable_or_ineligible": len(events) - len(exact),
        }
    )
    for difference, count in sorted(histogram.items()):
        SOURCE_EXACT_DISTRIBUTION.append(
            {
                "cohort": cohort,
                "expiration_minus_IEM_high_F": format(difference.normalize(), "f"),
                "event_count": count,
            }
        )

TWC_EVENTS = [event for event in EVENT_ROWS if cohort_from_primary_rules(event) == "KX citing TWC"]
TWC_NONZERO_DIFFERENCES = [
    {
        key: event[key]
        for key in [
            "event_ticker",
            "yes_ticker",
            "climate_date",
            "station",
            "expiration_value",
            "cli_max",
            "exact_difference_f",
            "cli_text",
        ]
    }
    for event in TWC_EVENTS
    if event["exact_difference_f"] not in (None, Decimal(0))
]
TWC_EXACT_UNAVAILABLE = []
for event in TWC_EVENTS:
    if event["exact_difference_f"] is None:
        TWC_EXACT_UNAVAILABLE.append(
            {
                "event_ticker": event["event_ticker"],
                "climate_date": event["climate_date"],
                "expiration_value_verbatim": sorted(
                    {str(row["expiration_raw"]) for row in GROUPED_MARKETS[event["event_ticker"]]}
                ),
                "reason": "No single numeric expiration_value; not an exact-match observation",
            }
        )

print("Source split: source comes from each market’s own rules_primary:")
show_table(SOURCE_COHORT_SUMMARY)
print("Exact differences by source (event weighted; Decimal zero formats are consolidated):")
show_table(SOURCE_EXACT_DISTRIBUTION)
print("Every TWC nonzero exact difference:")
show_table(TWC_NONZERO_DIFFERENCES)
print("TWC events without an exact comparison:")
show_table(TWC_EXACT_UNAVAILABLE)
print(
    "Legacy HIGH includes 2,934 events naming NWS in primary rules and 475 whose primary "
    "rules do not name a source. They remain separately audited in SOURCE_COUNTS; "
    "the control-family aggregate does not infer a missing primary source."
)


ACCOUNTING_CATEGORIES = (
    "compared",
    "zero_YES",
    "multiple_YES",
    "missing_CLI",
    "calendar_gap",
    "malformed_date",
)


def accounting_category(event):
    """Choose one exclusion reason; preserve all overlapping flags elsewhere."""
    if event["date_anomaly"]:
        return "malformed_date"
    if event["cli_missing_reason"] is not None:
        return "missing_CLI"
    if event["yes_count"] == 0:
        return "zero_YES"
    if event["yes_count"] > 1:
        return "multiple_YES"
    assert event["comparable"], event["event_ticker"]
    return "compared"


ACCOUNTING_ASSIGNMENTS = [
    {
        "series": event["series"],
        "unit": ("event", event["event_ticker"]),
        "category": accounting_category(event),
    }
    for event in EVENT_ROWS
]
ACCOUNTING_ASSIGNMENTS.extend(
    {
        "series": gap["series"],
        "unit": ("gap", gap["series"], gap["climate_date"]),
        "category": "calendar_gap",
    }
    for gap in GAPS
)
EVENT_ACCOUNTING = []
for series in SERIES_SUMMARY:
    assignments = [row for row in ACCOUNTING_ASSIGNMENTS if row["series"] == series["series"]]
    counts = Counter(row["category"] for row in assignments)
    EVENT_ACCOUNTING.append(
        {
            "series": series["series"],
            "actual_settled_events": series["events"],
            "total_calendar_event_slots": (series["last_date"] - series["first_date"]).days + 1,
            **{category: counts[category] for category in ACCOUNTING_CATEGORIES},
        }
    )

OVERLAPPING_EXCLUSION_FLAGS = []
for event in EVENT_ROWS:
    flags = []
    if event["date_anomaly"]:
        flags.append("malformed_date")
    if event["cli_missing_reason"] is not None:
        flags.append("missing_CLI")
    if event["yes_count"] == 0:
        flags.append("zero_YES")
    if event["yes_count"] > 1:
        flags.append("multiple_YES")
    if len(flags) > 1:
        OVERLAPPING_EXCLUSION_FLAGS.append(
            {
                "event_ticker": event["event_ticker"],
                "all_flags": flags,
                "counted_once_under": accounting_category(event),
            }
        )
print(
    "Per-series accounting: slots = compared + zero YES + multiple YES + missing CLI "
    "+ calendar gap + malformed date:"
)
show_table(EVENT_ACCOUNTING)
print("Actual settled events = total slots − calendar gaps. A calendar gap is not an event.")
print(
    "Exclusive precedence: malformed date, missing CLI, zero YES, multiple YES, compared. "
    "All original diagnostics remain in the earlier anomaly tables."
)
show_table(OVERLAPPING_EXCLUSION_FLAGS)


def describe_multiple_yes(event):
    markets = GROUPED_MARKETS[event["event_ticker"]]
    all_above = all(
        row["interval"][0] is not None and row["interval"][1] is None for row in markets
    )
    return {
        "event_ticker": event["event_ticker"],
        "series": event["series"],
        "market_count": len(markets),
        "yes_count": event["yes_count"],
        "classification": ("Non-exclusive above-X thresholds" if all_above else "Other structure"),
        "predicates": [row["bracket"] for row in markets],
        "event_mutually_exclusive_field": EVENTS[event["event_ticker"]]["mutually_exclusive"],
    }


MULTIPLE_YES_STRUCTURES = [
    describe_multiple_yes(event) for event in EVENT_ROWS if event["yes_count"] > 1
]
MULTIPLE_YES_EXAMPLES = []
for series in ["HIGHCHI", "HIGHNY"]:
    structure = next(row for row in MULTIPLE_YES_STRUCTURES if row["series"] == series)
    for market in GROUPED_MARKETS[structure["event_ticker"]]:
        MULTIPLE_YES_EXAMPLES.append(
            {
                "event_ticker": structure["event_ticker"],
                "ticker": market["ticker"],
                "predicate": market["bracket"],
                "result": market["result"],
                "rules_primary_verbatim": market["rules_primary"],
            }
        )
print("Multiple-YES structures (explanation only; comparison eligibility is unchanged):")
show_table(
    [
        {"classification": kind, "events": count}
        for kind, count in Counter(row["classification"] for row in MULTIPLE_YES_STRUCTURES).items()
    ]
)
show_table(MULTIPLE_YES_EXAMPLES)

twc = next(row for row in SOURCE_COHORT_SUMMARY if row["cohort"] == "KX citing TWC")
legacy = next(row for row in SOURCE_COHORT_SUMMARY if row["cohort"] == "Legacy HIGH")
RECORDED_DECISION_LINE = (
    "- 2026-11-12: Refines the 2026-11-07 settlement-source decision: all seven configured "
    "KX daily-high series cited NWS CLI through climate day 2026-08-13 and TWC from "
    f"2026-08-14; across {twc['total_events']} TWC events, TWC versus IEM CLI agrees on "
    f"{twc['bracket_matches']}/{twc['bracket_compared']} YES brackets and "
    f"{twc['exact_matches']}/{twc['exact_compared']} numeric exact maxima "
    "(one +5°F difference, KXHIGHMIA-26AUG29; one nonnumeric expiration excluded from "
    f"exact comparison); legacy HIGH control agrees on {legacy['bracket_matches']}/"
    f"{legacy['bracket_compared']} comparable brackets. Labels policy: Kalshi's settled "
    "result wherever a market exists, IEM's stored CLI high otherwise. Known limitation: "
    "IEM stores the latest issuance, which can differ from what was published at "
    "settlement (KXHIGHMIA-24NOV27 settled at 84°F before the CLI revised to 81°F). "
    "Evidence: notebooks/01_settlement_source.ipynb."
)
print("Decision recorded in docs/decisions.md:")
print(RECORDED_DECISION_LINE)

Source split: source comes from each market’s own rules_primary:


cohort,total_events,bracket_compared,bracket_matches,bracket_disagreements,bracket_agreement_pct,exact_compared,exact_matches,exact_nonzero,exact_agreement_pct,exact_unavailable_or_ineligible
KX citing NWS CLI,4486,4475,4474,1,99.9777,4015,4007,8,99.8007,471
KX citing TWC,630,630,629,1,99.8413,629,628,1,99.8410,1
Legacy HIGH,3409,3088,3088,0,100.0000,3407,3400,7,99.7945,2


Exact differences by source (event weighted; Decimal zero formats are consolidated):


cohort,expiration_minus_IEM_high_F,event_count
KX citing NWS CLI,-1,4
KX citing NWS CLI,0,4007
KX citing NWS CLI,1,3
KX citing NWS CLI,3,1
KX citing TWC,0,628
KX citing TWC,5,1
Legacy HIGH,-2,1
Legacy HIGH,-1,4
Legacy HIGH,0,3400
Legacy HIGH,1,1


Every TWC nonzero exact difference:


event_ticker,yes_ticker,climate_date,station,expiration_value,cli_max,exact_difference_f,cli_text
KXHIGHMIA-26AUG29,KXHIGHMIA-26AUG29-B90.5,2026-08-29,KMIA,90.00,85,5.00,https://mesonet.agron.iastate.edu/api/1/nwstext/202608300910-KMFL-CDUS42-CLIMIA


... (6 outputs left out)

Multiple-YES structures (explanation only; comparison eligibility is unchanged):


classification,events
Non-exclusive above-X thresholds,138


event_ticker,ticker,predicate,result,rules_primary_verbatim
HIGHCHI-21DEC02,HIGHCHI-21DEC02-T58,x > 58°F,yes,"If the highest temperature recorded at Chicago Midway, IL for December 02, 2021, is strictly greater than 58°, then the market resolves to Yes."
HIGHCHI-21DEC02,HIGHCHI-21DEC02-T56,x > 56°F,yes,"If the highest temperature recorded at Chicago Midway, IL for December 02, 2021, is strictly greater than 56°, then the market resolves to Yes."
HIGHNY-21DEC01,HIGHNY-21DEC01-T46,x > 46°F,yes,"If the highest temperature recorded in Central Park, New York for December 01, 2021, is strictly greater than 46°, then the market resolves to Yes."
HIGHNY-21DEC01,HIGHNY-21DEC01-T44,x > 44°F,yes,"If the highest temperature recorded in Central Park, New York for December 01, 2021, is strictly greater than 44°, then the market resolves to Yes."


Decision recorded in docs/decisions.md:
- 2026-11-12: Refines the 2026-11-07 settlement-source decision: all seven configured KX daily-high series cited NWS CLI through climate day 2026-08-13 and TWC from 2026-08-14; across 630 TWC events, TWC versus IEM CLI agrees on 629/630 YES brackets and 628/629 numeric exact maxima (one +5°F difference, KXHIGHMIA-26AUG29; one nonnumeric expiration excluded from exact comparison); legacy HIGH control agrees on 3088/3088 comparable brackets. Labels policy: Kalshi's settled result wherever a market exists, IEM's stored CLI high otherwise. Known limitation: IEM stores the latest issuance, which can differ from what was published at settlement (KXHIGHMIA-24NOV27 settled at 84°F before the CLI revised to 81°F). Evidence: notebooks/01_settlement_source.ipynb.


## Follow-up assertions and demonstrated failures

The added checks cover the source split, exclusive per-series accounting, actual
multiple-YES structures, and the recorded policy line. Each fault demonstration
temporarily replaces a report reader, runs the unchanged check, prints its failure,
restores the original reader, and passes again. The decision-file demonstration
alters only the in-memory reader; the correct on-disk decision is preserved.

In [11]:
def source_summary_for_validation():
    return SOURCE_COHORT_SUMMARY


def source_distribution_for_validation():
    return SOURCE_EXACT_DISTRIBUTION


def event_accounting_for_validation():
    return EVENT_ACCOUNTING


def multiple_yes_structures_for_validation():
    return MULTIPLE_YES_STRUCTURES


def recorded_decision_lines_for_validation():
    text = (ROOT / "docs/decisions.md").read_text()
    settled = text.split("## Settled\n", 1)[1].split("\n## Open", 1)[0]
    return [
        line
        for line in settled.splitlines()
        if line.startswith("- 2026-11-12: Refines the 2026-11-07 settlement-source decision:")
    ]


def check_source_summaries_preserve_each_primary_rule_cohort_and_exact_difference():
    """Source summaries preserve each primary-rule cohort and every exact difference."""
    expected = defaultdict(list)
    for event in EVENT_ROWS:
        markets = GROUPED_MARKETS[event["event_ticker"]]
        raw_kx = all(market["ticker"].startswith("KX") for market in markets)
        if not raw_kx:
            assert all(market["ticker"].startswith("HIGH") for market in markets)
            cohort = "Legacy HIGH"
        else:
            names = set()
            for market in markets:
                primary = RAW_BY_TICKER[market["ticker"]]["rules_primary"].casefold()
                if "the weather company" in primary:
                    names.add("KX citing TWC")
                else:
                    assert "national weather service" in primary or re.search(r"\bnws\b", primary)
                    names.add("KX citing NWS CLI")
            assert len(names) == 1
            cohort = next(iter(names))
        expected[cohort].append(event)
    reported = source_summary_for_validation()
    assert len(reported) == len(expected) == 3
    assert {row["cohort"] for row in reported} == expected.keys()
    expected_distribution = Counter()
    for row in reported:
        events = expected[row["cohort"]]
        compared = [event for event in events if event["comparable"]]
        exact = [event for event in events if event["exact_difference_f"] is not None]
        assert row["total_events"] == len(events), (
            "A market was assigned to the wrong source cohort"
        )
        assert row["bracket_compared"] == len(compared)
        assert row["bracket_matches"] == sum(event["disagreement"] is False for event in compared)
        assert row["bracket_disagreements"] == sum(
            event["disagreement"] is True for event in compared
        )
        assert row["exact_compared"] == len(exact)
        assert row["exact_matches"] == sum(event["exact_difference_f"] == 0 for event in exact)
        assert row["exact_nonzero"] == sum(event["exact_difference_f"] != 0 for event in exact)
        for event in exact:
            expected_distribution[row["cohort"], event["exact_difference_f"]] += 1
    actual_distribution = Counter(
        {
            (row["cohort"], Decimal(row["expiration_minus_IEM_high_F"])): row["event_count"]
            for row in source_distribution_for_validation()
        }
    )
    assert actual_distribution == expected_distribution, (
        "Source-specific exact differences were lost"
    )
    twc_expected = {
        event["event_ticker"]: event["exact_difference_f"]
        for event in expected["KX citing TWC"]
        if event["exact_difference_f"] not in (None, Decimal(0))
    }
    assert {
        row["event_ticker"]: row["exact_difference_f"] for row in TWC_NONZERO_DIFFERENCES
    } == twc_expected
    assert sum(row["total_events"] for row in reported) == len(EVENT_ROWS)
    assert sum(row["bracket_compared"] for row in reported) == sum(
        event["comparable"] for event in EVENT_ROWS
    )


def check_each_series_accounts_for_every_event_and_calendar_gap_once():
    """Each series accounts for every event and calendar gap exactly once."""
    expected_units = {("event", event["event_ticker"]) for event in EVENT_ROWS}
    expected_units.update(("gap", gap["series"], gap["climate_date"]) for gap in GAPS)
    assert len(ACCOUNTING_ASSIGNMENTS) == len(expected_units)
    assert {row["unit"] for row in ACCOUNTING_ASSIGNMENTS} == expected_units
    events_by_id = {event["event_ticker"]: event for event in EVENT_ROWS}
    for assignment in ACCOUNTING_ASSIGNMENTS:
        if assignment["unit"][0] == "gap":
            assert assignment["category"] == "calendar_gap"
            continue
        event = events_by_id[assignment["unit"][1]]
        flags = {
            "malformed_date": event["date_anomaly"],
            "missing_CLI": not event["date_anomaly"] and event["cli_missing_reason"] is not None,
            "zero_YES": not event["date_anomaly"]
            and event["cli_missing_reason"] is None
            and event["yes_count"] == 0,
            "multiple_YES": not event["date_anomaly"]
            and event["cli_missing_reason"] is None
            and event["yes_count"] > 1,
            "compared": event["comparable"],
        }
        assert sum(flags.values()) == 1, ("Non-exclusive accounting", event["event_ticker"])
        assert flags[assignment["category"]], ("Wrong accounting category", event["event_ticker"])
    accounting = event_accounting_for_validation()
    assert len(accounting) == len(SERIES_SUMMARY)
    assert {row["series"] for row in accounting} == {row["series"] for row in SERIES_SUMMARY}
    for row in accounting:
        series = next(series for series in SERIES_SUMMARY if series["series"] == row["series"])
        assert row["actual_settled_events"] == series["events"], (
            "Calendar gaps are not settled events"
        )
        assert (
            sum(row[category] for category in ACCOUNTING_CATEGORIES)
            == (row["total_calendar_event_slots"])
        ), "Accounting categories do not sum to the per-series total"
        assert (
            row["total_calendar_event_slots"] == row["actual_settled_events"] + row["calendar_gap"]
        )
        assert row["compared"] == series["comparable_events"], "Comparison eligibility changed"
        assert all(
            type(row[category]) is int and row[category] >= 0 for category in ACCOUNTING_CATEGORIES
        )
        actual_counts = Counter(
            assignment["category"]
            for assignment in ACCOUNTING_ASSIGNMENTS
            if assignment["series"] == row["series"]
        )
        assert all(row[category] == actual_counts[category] for category in ACCOUNTING_CATEGORIES)


def check_multiple_yes_explanations_follow_the_actual_market_predicates():
    """Multiple-YES explanations follow actual predicates and retain two example events."""
    expected = {event["event_ticker"] for event in EVENT_ROWS if event["yes_count"] > 1}
    reported = multiple_yes_structures_for_validation()
    assert len(reported) == len(expected)
    assert {row["event_ticker"] for row in reported} == expected
    for row in reported:
        markets = GROUPED_MARKETS[row["event_ticker"]]
        # Unbounded upper tails intersect whenever x exceeds every lower bound;
        # several YES results therefore do not imply mutually exclusive brackets.
        all_above = all(
            interval_from_rules(market["rules_primary"])[1] is None
            and interval_from_rules(market["rules_primary"])[0] is not None
            for market in markets
        )
        expected_kind = "Non-exclusive above-X thresholds" if all_above else "Other structure"
        assert row["classification"] == expected_kind, (
            "Multiple-YES market structure was misclassified"
        )
        assert row["predicates"] == [market["bracket"] for market in markets]
    assert len({row["event_ticker"] for row in MULTIPLE_YES_EXAMPLES}) == 2
    for example in MULTIPLE_YES_EXAMPLES:
        assert (
            example["rules_primary_verbatim"] == RAW_BY_TICKER[example["ticker"]]["rules_primary"]
        )
        assert example["event_ticker"] in expected


def check_the_recorded_decision_matches_the_frozen_source_evidence_and_label_policy():
    """The recorded decision matches frozen source evidence and the approved label policy."""
    lines = recorded_decision_lines_for_validation()
    assert len(lines) == 1, "The settled decision is missing, duplicated, or outside Settled"
    line = lines[0]
    counts = re.search(
        r"agrees on (\d+)/(\d+) YES brackets and (\d+)/(\d+) numeric exact maxima", line
    )
    assert counts, "The decision omits the numeric TWC evidence"
    twc = [
        event
        for event in EVENT_ROWS
        if event["family"] == "KX" and event["source"] == "The Weather Company"
    ]
    compared = [event for event in twc if event["comparable"]]
    numeric = [event for event in twc if event["exact_difference_f"] is not None]
    expected = (
        sum(event["disagreement"] is False for event in compared),
        len(compared),
        sum(event["exact_difference_f"] == 0 for event in numeric),
        len(numeric),
    )
    assert tuple(map(int, counts.groups())) == expected, (
        "Decision numbers disagree with source evidence"
    )
    assert f"across {len(twc)} TWC events" in line
    legacy_compared = [
        event for event in EVENT_ROWS if event["family"] == "HIGH" and event["comparable"]
    ]
    assert (
        f"legacy HIGH control agrees on "
        f"{sum(event['disagreement'] is False for event in legacy_compared)}/"
        f"{len(legacy_compared)} comparable brackets"
    ) in line
    for station in STATIONS:
        markets = [row for row in NORMALIZED if row["series"] == station["series_ticker"]]
        last_nws = max(row["climate_date"] for row in markets if row["source"] == "NWS CLI")
        first_twc = min(
            row["climate_date"] for row in markets if row["source"] == "The Weather Company"
        )
        assert f"NWS CLI through climate day {last_nws} and TWC from {first_twc}" in line
    assert (
        "Kalshi's settled result wherever a market exists, IEM's stored CLI high otherwise" in line
    )
    assert "IEM stores the latest issuance" in line and "KXHIGHMIA-24NOV27" in line
    assert line == RECORDED_DECISION_LINE


FOLLOWUP_CHECKS = [
    check_source_summaries_preserve_each_primary_rule_cohort_and_exact_difference,
    check_each_series_accounts_for_every_event_and_calendar_gap_once,
    check_multiple_yes_explanations_follow_the_actual_market_predicates,
    check_the_recorded_decision_matches_the_frozen_source_evidence_and_label_policy,
]
for index, check in enumerate(FOLLOWUP_CHECKS, start=11):
    check()
    print(f"PASS {index}: {check.__doc__}")


def incorrectly_grouped_source_summary():
    rows = [dict(row) for row in SOURCE_COHORT_SUMMARY]
    next(row for row in rows if row["cohort"] == "KX citing NWS CLI")["total_events"] -= 1
    next(row for row in rows if row["cohort"] == "KX citing TWC")["total_events"] += 1
    return rows


def double_count_the_missing_zero_yes_event():
    rows = [dict(row) for row in EVENT_ACCOUNTING]
    next(row for row in rows if row["series"] == "KXHIGHMIA")["zero_YES"] += 1
    return rows


def call_calendar_gaps_settled_events():
    rows = [dict(row) for row in EVENT_ACCOUNTING]
    for row in rows:
        row["actual_settled_events"] += row["calendar_gap"]
    return rows


def incorrectly_explained_multiple_yes_event():
    rows = [dict(row) for row in MULTIPLE_YES_STRUCTURES]
    rows[0]["classification"] = "Other structure"
    return rows


real_decision_lines = recorded_decision_lines_for_validation
FOLLOWUP_PROOFS = [
    prove_failure(
        "11a",
        "source_summary_for_validation",
        incorrectly_grouped_source_summary,
        FOLLOWUP_CHECKS[0],
        "Move a real NWS event into the TWC cohort.",
    ),
    prove_failure(
        "11b",
        "source_distribution_for_validation",
        lambda: [
            row
            for row in SOURCE_EXACT_DISTRIBUTION
            if not (
                row["cohort"] == "KX citing TWC"
                and Decimal(row["expiration_minus_IEM_high_F"]) != 0
            )
        ],
        FOLLOWUP_CHECKS[0],
        "Suppress the TWC nonzero exact difference.",
    ),
    prove_failure(
        "12a",
        "event_accounting_for_validation",
        double_count_the_missing_zero_yes_event,
        FOLLOWUP_CHECKS[1],
        "Count the Miami missing-CLI zero-YES event twice.",
    ),
    prove_failure(
        "12b",
        "event_accounting_for_validation",
        call_calendar_gaps_settled_events,
        FOLLOWUP_CHECKS[1],
        "Mislabel absent calendar dates as settled events.",
    ),
    prove_failure(
        "13",
        "multiple_yes_structures_for_validation",
        incorrectly_explained_multiple_yes_event,
        FOLLOWUP_CHECKS[2],
        "Describe a real nested-threshold event as another structure.",
    ),
    prove_failure(
        "14",
        "recorded_decision_lines_for_validation",
        lambda: [
            line.replace("628/629 numeric exact maxima", "629/629 numeric exact maxima")
            for line in real_decision_lines()
        ],
        FOLLOWUP_CHECKS[3],
        "Claim the one TWC exact mismatch was an exact match.",
    ),
]
show_table(FOLLOWUP_PROOFS)
assert all(row["cache"] for row in REQUESTS[FOLLOWUP_REQUEST_COUNT:]), (
    "The follow-up must use frozen cached data"
)
print(
    f"Follow-up complete: {len(CHECKS) + len(FOLLOWUP_CHECKS)} checks passed; "
    f"{len(PROOFS) + len(FOLLOWUP_PROOFS)} demonstrated failures restored; "
    f"{sum(not row['cache'] for row in REQUESTS)} network requests."
)

PASS 11: Source summaries preserve each primary-rule cohort and every exact difference.
PASS 12: Each series accounts for every event and calendar gap exactly once.
PASS 13: Multiple-YES explanations follow actual predicates and retain two example events.


PASS 14: The recorded decision matches frozen source evidence and the approved label policy.
EXPECTED FAILURE 11a: Move a real NWS event into the TWC cohort.
AssertionError: A market was assigned to the wrong source cohort


RESTORED PASS 11a: Source summaries preserve each primary-rule cohort and every exact difference.

EXPECTED FAILURE 11b: Suppress the TWC nonzero exact difference.
AssertionError: Source-specific exact differences were lost


RESTORED PASS 11b: Source summaries preserve each primary-rule cohort and every exact difference.


EXPECTED FAILURE 12a: Count the Miami missing-CLI zero-YES event twice.
AssertionError: Accounting categories do not sum to the per-series total
RESTORED PASS 12a: Each series accounts for every event and calendar gap exactly once.
EXPECTED FAILURE 12b: Mislabel absent calendar dates as settled events.
AssertionError: Calendar gaps are not settled events
RESTORED PASS 12b: Each series accounts for every event and calendar gap exactly once.
EXPECTED FAILURE 13: Describe a real nested-threshold event as another structure.
AssertionError: Multiple-YES market structure was misclassified
RESTORED PASS 13: Multiple-YES explanations follow actual predicates and retain two example events.
EXPECTED FAILURE 14: Claim the one TWC exact mismatch was an exact match.
AssertionError: Decision numbers disagree with source evidence


RESTORED PASS 14: The recorded decision matches frozen source evidence and the approved label policy.


property,check,violation,observed_failure,restored
11a,check_source_summaries_preserve_each_primary_rule_cohort_and_exact_difference,Move a real NWS event into the TWC cohort.,AssertionError: A market was assigned to the wrong source cohort,PASS
11b,check_source_summaries_preserve_each_primary_rule_cohort_and_exact_difference,Suppress the TWC nonzero exact difference.,AssertionError: Source-specific exact differences were lost,PASS
12a,check_each_series_accounts_for_every_event_and_calendar_gap_once,Count the Miami missing-CLI zero-YES event twice.,AssertionError: Accounting categories do not sum to the per-series total,PASS
12b,check_each_series_accounts_for_every_event_and_calendar_gap_once,Mislabel absent calendar dates as settled events.,AssertionError: Calendar gaps are not settled events,PASS
13,check_multiple_yes_explanations_follow_the_actual_market_predicates,Describe a real nested-threshold event as another structure.,AssertionError: Multiple-YES market structure was misclassified,PASS
14,check_the_recorded_decision_matches_the_frozen_source_evidence_and_label_policy,Claim the one TWC exact mismatch was an exact match.,AssertionError: Decision numbers disagree with source evidence,PASS


Follow-up complete: 14 checks passed; 18 demonstrated failures restored; 0 network requests.
